<a href="https://colab.research.google.com/github/anuvishalp/SQL-DataEngg-DB/blob/main/SqlPython%20QnA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Provide multiple solutions for all these problems using oracle sql,plsql,python 175. Combine Two Tables: Join two tables to get information from both.
    176. Second Highest Salary: Find the second highest salary from an employee table.
    181. Employees Earning More Than Their Managers: Identify employees whose salary is greater than their manager's salary.
    183. Customers Who Never Order: Find customers who have not placed any orders.
    197. Rising Temperature: Determine days when the temperature was higher than the previous day.
    595. Big Countries: Select countries with a large area or population.
    607. Sales Person: Find salespersons who did not make any sales to a specific company.
    620. Not Boring Movies: Select movies with odd IDs and a description that is not 'boring'.
    1068. Product Sales Analysis I: Analyze sales data for products.
    1322. Ads Performance: Calculate the click-through rate for advertisements.
    1378. Replace Employee ID With The Unique Identifier: Replace employee IDs with their unique identifiers.

In [ ]:
-- ============================================================================
-- 175. Combine Two Tables
-- ============================================================================
-- Tables: Person (personId, lastName, firstName)
--         Address (addressId, personId, city, state)

-- Solution 1: LEFT JOIN
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person p
LEFT JOIN Address a ON p.personId = a.personId;

-- Solution 2: Using Oracle's (+) syntax
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person p, Address a
WHERE p.personId = a.personId(+);

-- Solution 3: LEFT OUTER JOIN with USING
SELECT firstName, lastName, city, state
FROM Person
LEFT OUTER JOIN Address USING (personId);

-- PL/SQL Solution
DECLARE
    CURSOR person_cursor IS
        SELECT p.firstName, p.lastName, a.city, a.state
        FROM Person p
        LEFT JOIN Address a ON p.personId = a.personId;
BEGIN
    FOR rec IN person_cursor LOOP
        DBMS_OUTPUT.PUT_LINE(rec.firstName || ' ' || rec.lastName ||
                           ' - ' || NVL(rec.city, 'NULL') ||
                           ', ' || NVL(rec.state, 'NULL'));
    END LOOP;
END;
/

-- ============================================================================
-- 176. Second Highest Salary
-- ============================================================================
-- Table: Employee (id, salary)

-- Solution 1: Using OFFSET-FETCH (Oracle 12c+)
SELECT DISTINCT salary AS SecondHighestSalary
FROM Employee
ORDER BY salary DESC
OFFSET 1 ROW FETCH NEXT 1 ROW ONLY;

-- Solution 2: Using subquery with MAX
SELECT MAX(salary) AS SecondHighestSalary
FROM Employee
WHERE salary < (SELECT MAX(salary) FROM Employee);

-- Solution 3: Using ROW_NUMBER
SELECT salary AS SecondHighestSalary
FROM (
    SELECT salary, ROW_NUMBER() OVER (ORDER BY salary DESC) AS rn
    FROM (SELECT DISTINCT salary FROM Employee)
)
WHERE rn = 2;

-- Solution 4: Using DENSE_RANK
SELECT DISTINCT salary AS SecondHighestSalary
FROM (
    SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM Employee
)
WHERE rnk = 2;

-- Solution 5: Handle NULL when no second highest exists
SELECT NVL(
    (SELECT DISTINCT salary
     FROM Employee
     ORDER BY salary DESC
     OFFSET 1 ROW FETCH NEXT 1 ROW ONLY),
    NULL
) AS SecondHighestSalary
FROM DUAL;

-- PL/SQL Solution
DECLARE
    v_second_highest NUMBER;
BEGIN
    SELECT MAX(salary) INTO v_second_highest
    FROM Employee
    WHERE salary < (SELECT MAX(salary) FROM Employee);

    DBMS_OUTPUT.PUT_LINE('Second Highest Salary: ' || NVL(TO_CHAR(v_second_highest), 'NULL'));
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        DBMS_OUTPUT.PUT_LINE('Second Highest Salary: NULL');
END;
/

-- ============================================================================
-- 181. Employees Earning More Than Their Managers
-- ============================================================================
-- Table: Employee (id, name, salary, managerId)

-- Solution 1: Self JOIN
SELECT e1.name AS Employee
FROM Employee e1
JOIN Employee e2 ON e1.managerId = e2.id
WHERE e1.salary > e2.salary;

-- Solution 2: Using subquery
SELECT e1.name AS Employee
FROM Employee e1
WHERE e1.salary > (
    SELECT e2.salary
    FROM Employee e2
    WHERE e2.id = e1.managerId
);

-- Solution 3: Using EXISTS
SELECT e1.name AS Employee
FROM Employee e1
WHERE EXISTS (
    SELECT 1
    FROM Employee e2
    WHERE e2.id = e1.managerId
    AND e1.salary > e2.salary
);

-- PL/SQL Solution
DECLARE
    CURSOR emp_cursor IS
        SELECT e1.name
        FROM Employee e1
        JOIN Employee e2 ON e1.managerId = e2.id
        WHERE e1.salary > e2.salary;
BEGIN
    FOR rec IN emp_cursor LOOP
        DBMS_OUTPUT.PUT_LINE('Employee: ' || rec.name);
    END LOOP;
END;
/

-- ============================================================================
-- 183. Customers Who Never Order
-- ============================================================================
-- Tables: Customers (id, name), Orders (id, customerId)

-- Solution 1: LEFT JOIN with NULL check
SELECT c.name AS Customers
FROM Customers c
LEFT JOIN Orders o ON c.id = o.customerId
WHERE o.id IS NULL;

-- Solution 2: NOT IN subquery
SELECT name AS Customers
FROM Customers
WHERE id NOT IN (SELECT customerId FROM Orders WHERE customerId IS NOT NULL);

-- Solution 3: NOT EXISTS
SELECT name AS Customers
FROM Customers c
WHERE NOT EXISTS (
    SELECT 1 FROM Orders o WHERE o.customerId = c.id
);

-- Solution 4: MINUS/EXCEPT operator
SELECT name AS Customers
FROM Customers
WHERE id IN (
    SELECT id FROM Customers
    MINUS
    SELECT customerId FROM Orders
);

-- PL/SQL Solution
DECLARE
    CURSOR no_order_cursor IS
        SELECT c.name
        FROM Customers c
        WHERE NOT EXISTS (
            SELECT 1 FROM Orders o WHERE o.customerId = c.id
        );
BEGIN
    FOR rec IN no_order_cursor LOOP
        DBMS_OUTPUT.PUT_LINE('Customer: ' || rec.name);
    END LOOP;
END;
/

-- ============================================================================
-- 197. Rising Temperature
-- ============================================================================
-- Table: Weather (id, recordDate, temperature)

-- Solution 1: Self JOIN with date comparison
SELECT w1.id
FROM Weather w1
JOIN Weather w2 ON w1.recordDate = w2.recordDate + INTERVAL '1' DAY
WHERE w1.temperature > w2.temperature;

-- Solution 2: Using LAG window function
SELECT id
FROM (
    SELECT id, temperature,
           LAG(temperature) OVER (ORDER BY recordDate) AS prev_temp,
           recordDate,
           LAG(recordDate) OVER (ORDER BY recordDate) AS prev_date
    FROM Weather
)
WHERE temperature > prev_temp
AND recordDate = prev_date + INTERVAL '1' DAY;

-- Solution 3: Subquery approach
SELECT w1.id
FROM Weather w1
WHERE w1.temperature > (
    SELECT w2.temperature
    FROM Weather w2
    WHERE w2.recordDate = w1.recordDate - INTERVAL '1' DAY
);

-- PL/SQL Solution
DECLARE
    CURSOR weather_cursor IS
        SELECT w1.id
        FROM Weather w1
        JOIN Weather w2 ON w1.recordDate = w2.recordDate + INTERVAL '1' DAY
        WHERE w1.temperature > w2.temperature;
BEGIN
    FOR rec IN weather_cursor LOOP
        DBMS_OUTPUT.PUT_LINE('ID: ' || rec.id);
    END LOOP;
END;
/

-- ============================================================================
-- 595. Big Countries
-- ============================================================================
-- Table: World (name, continent, area, population, gdp)

-- Solution 1: Using OR condition
SELECT name, population, area
FROM World
WHERE area >= 3000000 OR population >= 25000000;

-- Solution 2: Using UNION
SELECT name, population, area
FROM World
WHERE area >= 3000000
UNION
SELECT name, population, area
FROM World
WHERE population >= 25000000;

-- Solution 3: Using IN with subquery
SELECT name, population, area
FROM World
WHERE name IN (
    SELECT name FROM World WHERE area >= 3000000
    UNION
    SELECT name FROM World WHERE population >= 25000000
);

-- PL/SQL Solution
DECLARE
    CURSOR big_countries_cursor IS
        SELECT name, population, area
        FROM World
        WHERE area >= 3000000 OR population >= 25000000;
BEGIN
    FOR rec IN big_countries_cursor LOOP
        DBMS_OUTPUT.PUT_LINE(rec.name || ' - Pop: ' || rec.population ||
                           ', Area: ' || rec.area);
    END LOOP;
END;
/

-- ============================================================================
-- 607. Sales Person
-- ============================================================================
-- Tables: SalesPerson (sales_id, name)
--         Company (com_id, name)
--         Orders (order_id, order_date, com_id, sales_id, amount)

-- Solution 1: NOT IN with subquery
SELECT name
FROM SalesPerson
WHERE sales_id NOT IN (
    SELECT DISTINCT o.sales_id
    FROM Orders o
    JOIN Company c ON o.com_id = c.com_id
    WHERE c.name = 'RED'
);

-- Solution 2: NOT EXISTS
SELECT sp.name
FROM SalesPerson sp
WHERE NOT EXISTS (
    SELECT 1
    FROM Orders o
    JOIN Company c ON o.com_id = c.com_id
    WHERE c.name = 'RED' AND o.sales_id = sp.sales_id
);

-- Solution 3: LEFT JOIN with NULL check
SELECT sp.name
FROM SalesPerson sp
LEFT JOIN (
    SELECT DISTINCT o.sales_id
    FROM Orders o
    JOIN Company c ON o.com_id = c.com_id
    WHERE c.name = 'RED'
) red_sales ON sp.sales_id = red_sales.sales_id
WHERE red_sales.sales_id IS NULL;

-- Solution 4: MINUS operator
SELECT name
FROM SalesPerson
WHERE sales_id IN (
    SELECT sales_id FROM SalesPerson
    MINUS
    SELECT sales_id FROM Orders o
    JOIN Company c ON o.com_id = c.com_id
    WHERE c.name = 'RED'
);

-- ============================================================================
-- 620. Not Boring Movies
-- ============================================================================
-- Table: Cinema (id, movie, description, rating)

-- Solution 1: Using MOD function
SELECT id, movie, description, rating
FROM Cinema
WHERE MOD(id, 2) = 1 AND description != 'boring'
ORDER BY rating DESC;

-- Solution 2: Using bitwise AND
SELECT id, movie, description, rating
FROM Cinema
WHERE BITAND(id, 1) = 1 AND description != 'boring'
ORDER BY rating DESC;

-- Solution 3: Using REGEXP
SELECT id, movie, description, rating
FROM Cinema
WHERE REGEXP_LIKE(id, '[13579]$') AND description != 'boring'
ORDER BY rating DESC;

-- PL/SQL Solution
DECLARE
    CURSOR movie_cursor IS
        SELECT id, movie, description, rating
        FROM Cinema
        WHERE MOD(id, 2) = 1 AND description != 'boring'
        ORDER BY rating DESC;
BEGIN
    FOR rec IN movie_cursor LOOP
        DBMS_OUTPUT.PUT_LINE(rec.id || ' - ' || rec.movie ||
                           ' (Rating: ' || rec.rating || ')');
    END LOOP;
END;
/

-- ============================================================================
-- 1068. Product Sales Analysis I
-- ============================================================================
-- Tables: Sales (sale_id, product_id, year, quantity, price)
--         Product (product_id, product_name)

-- Solution 1: INNER JOIN
SELECT p.product_name, s.year, s.price
FROM Sales s
JOIN Product p ON s.product_id = p.product_id;

-- Solution 2: Using subquery
SELECT
    (SELECT product_name FROM Product WHERE product_id = s.product_id) AS product_name,
    s.year,
    s.price
FROM Sales s;

-- Solution 3: Using NATURAL JOIN (if column names match)
SELECT product_name, year, price
FROM Sales
NATURAL JOIN Product;

-- PL/SQL Solution
DECLARE
    CURSOR sales_cursor IS
        SELECT p.product_name, s.year, s.price
        FROM Sales s
        JOIN Product p ON s.product_id = p.product_id;
BEGIN
    FOR rec IN sales_cursor LOOP
        DBMS_OUTPUT.PUT_LINE(rec.product_name || ' - Year: ' || rec.year ||
                           ', Price: ' || rec.price);
    END LOOP;
END;
/

-- ============================================================================
-- 1322. Ads Performance
-- ============================================================================
-- Table: Ads (ad_id, user_id, action) where action in ('Clicked','Viewed','Ignored')

-- Solution 1: Using CASE and aggregation
SELECT
    ad_id,
    ROUND(
        COALESCE(
            SUM(CASE WHEN action = 'Clicked' THEN 1 ELSE 0 END) * 100.0 /
            NULLIF(SUM(CASE WHEN action IN ('Clicked', 'Viewed') THEN 1 ELSE 0 END), 0),
            0
        ), 2
    ) AS ctr
FROM Ads
GROUP BY ad_id
ORDER BY ctr DESC, ad_id ASC;

-- Solution 2: Using filter clause (Oracle 12c+)
SELECT
    ad_id,
    ROUND(
        COALESCE(
            COUNT(*) FILTER (WHERE action = 'Clicked') * 100.0 /
            NULLIF(COUNT(*) FILTER (WHERE action IN ('Clicked', 'Viewed')), 0),
            0
        ), 2
    ) AS ctr
FROM Ads
GROUP BY ad_id
ORDER BY ctr DESC, ad_id ASC;

-- Solution 3: Using subqueries
SELECT
    ad_id,
    ROUND(
        COALESCE(
            (SELECT COUNT(*) FROM Ads a2
             WHERE a2.ad_id = a1.ad_id AND a2.action = 'Clicked') * 100.0 /
            NULLIF((SELECT COUNT(*) FROM Ads a3
                    WHERE a3.ad_id = a1.ad_id
                    AND a3.action IN ('Clicked', 'Viewed')), 0),
            0
        ), 2
    ) AS ctr
FROM (SELECT DISTINCT ad_id FROM Ads) a1
ORDER BY ctr DESC, ad_id ASC;

-- PL/SQL Solution
DECLARE
    CURSOR ads_cursor IS
        SELECT
            ad_id,
            ROUND(
                COALESCE(
                    SUM(CASE WHEN action = 'Clicked' THEN 1 ELSE 0 END) * 100.0 /
                    NULLIF(SUM(CASE WHEN action IN ('Clicked', 'Viewed') THEN 1 ELSE 0 END), 0),
                    0
                ), 2
            ) AS ctr
        FROM Ads
        GROUP BY ad_id
        ORDER BY ctr DESC, ad_id ASC;
BEGIN
    FOR rec IN ads_cursor LOOP
        DBMS_OUTPUT.PUT_LINE('Ad ID: ' || rec.ad_id || ', CTR: ' || rec.ctr || '%');
    END LOOP;
END;
/

-- ============================================================================
-- 1378. Replace Employee ID With The Unique Identifier
-- ============================================================================
-- Tables: Employees (id, name)
--         EmployeeUNI (id, unique_id)

-- Solution 1: LEFT JOIN
SELECT eu.unique_id, e.name
FROM Employees e
LEFT JOIN EmployeeUNI eu ON e.id = eu.id;

-- Solution 2: Using Oracle's (+) syntax
SELECT eu.unique_id, e.name
FROM Employees e, EmployeeUNI eu
WHERE e.id = eu.id(+);

-- Solution 3: Using OUTER APPLY (Oracle 12c+)
SELECT eu.unique_id, e.name
FROM Employees e
OUTER APPLY (
    SELECT unique_id
    FROM EmployeeUNI
    WHERE id = e.id
) eu;

-- PL/SQL Solution
DECLARE
    CURSOR emp_uni_cursor IS
        SELECT eu.unique_id, e.name
        FROM Employees e
        LEFT JOIN EmployeeUNI eu ON e.id = eu.id;
BEGIN
    FOR rec IN emp_uni_cursor LOOP
        DBMS_OUTPUT.PUT_LINE('Unique ID: ' || NVL(TO_CHAR(rec.unique_id), 'NULL') ||
                           ', Name: ' || rec.name);
    END LOOP;
END;
/

In [ ]:
import pandas as pd
import cx_Oracle
from datetime import datetime, timedelta

# ============================================================================
# 175. Combine Two Tables
# ============================================================================

def combine_two_tables_pandas(person_df, address_df):
    """Solution using Pandas merge"""
    result = person_df.merge(address_df, on='personId', how='left')
    return result[['firstName', 'lastName', 'city', 'state']]

def combine_two_tables_sql(connection):
    """Solution using direct SQL execution"""
    query = """
        SELECT p.firstName, p.lastName, a.city, a.state
        FROM Person p
        LEFT JOIN Address a ON p.personId = a.personId
    """
    return pd.read_sql(query, connection)

def combine_two_tables_dict(person_list, address_list):
    """Solution using dictionaries"""
    address_map = {addr['personId']: addr for addr in address_list}

    result = []
    for person in person_list:
        address = address_map.get(person['personId'], {})
        result.append({
            'firstName': person['firstName'],
            'lastName': person['lastName'],
            'city': address.get('city'),
            'state': address.get('state')
        })
    return result

# ============================================================================
# 176. Second Highest Salary
# ============================================================================

def second_highest_salary_pandas(employee_df):
    """Solution 1: Using Pandas nlargest"""
    unique_salaries = employee_df['salary'].drop_duplicates().nlargest(2)
    return unique_salaries.iloc[1] if len(unique_salaries) >= 2 else None

def second_highest_salary_sort(employee_df):
    """Solution 2: Using sort and indexing"""
    sorted_salaries = sorted(employee_df['salary'].unique(), reverse=True)
    return sorted_salaries[1] if len(sorted_salaries) >= 2 else None

def second_highest_salary_set(employee_list):
    """Solution 3: Using set and sorted"""
    salaries = sorted(set(emp['salary'] for emp in employee_list), reverse=True)
    return salaries[1] if len(salaries) >= 2 else None

def second_highest_salary_max(employee_df):
    """Solution 4: Using max with condition"""
    max_salary = employee_df['salary'].max()
    second_max = employee_df[employee_df['salary'] < max_salary]['salary'].max()
    return second_max if pd.notna(second_max) else None

# ============================================================================
# 181. Employees Earning More Than Their Managers
# ============================================================================

def employees_earning_more_pandas(employee_df):
    """Solution 1: Using Pandas merge"""
    result = employee_df.merge(
        employee_df,
        left_on='managerId',
        right_on='id',
        suffixes=('_emp', '_mgr')
    )
    result = result[result['salary_emp'] > result['salary_mgr']]
    return result[['name_emp']].rename(columns={'name_emp': 'Employee'})

def employees_earning_more_dict(employee_list):
    """Solution 2: Using dictionary lookup"""
    emp_map = {emp['id']: emp for emp in employee_list}

    result = []
    for emp in employee_list:
        if emp.get('managerId') and emp['managerId'] in emp_map:
            manager = emp_map[emp['managerId']]
            if emp['salary'] > manager['salary']:
                result.append(emp['name'])
    return result

def employees_earning_more_iterative(employee_df):
    """Solution 3: Using iterative approach"""
    result = []
    for idx, emp in employee_df.iterrows():
        if pd.notna(emp['managerId']):
            manager = employee_df[employee_df['id'] == emp['managerId']]
            if not manager.empty and emp['salary'] > manager.iloc[0]['salary']:
                result.append(emp['name'])
    return result

# ============================================================================
# 183. Customers Who Never Order
# ============================================================================

def customers_never_order_pandas(customers_df, orders_df):
    """Solution 1: Using Pandas merge with indicator"""
    merged = customers_df.merge(orders_df, left_on='id', right_on='customerId',
                                how='left', indicator=True)
    no_orders = merged[merged['_merge'] == 'left_only']
    return no_orders[['name']].rename(columns={'name': 'Customers'})

def customers_never_order_isin(customers_df, orders_df):
    """Solution 2: Using isin"""
    customer_ids_with_orders = orders_df['customerId'].unique()
    no_orders = customers_df[~customers_df['id'].isin(customer_ids_with_orders)]
    return no_orders[['name']].rename(columns={'name': 'Customers'})

def customers_never_order_set(customer_list, order_list):
    """Solution 3: Using set difference"""
    all_customer_ids = {c['id'] for c in customer_list}
    customer_ids_with_orders = {o['customerId'] for o in order_list}
    customers_without_orders = all_customer_ids - customer_ids_with_orders

    return [c['name'] for c in customer_list if c['id'] in customers_without_orders]

# ============================================================================
# 197. Rising Temperature
# ============================================================================

def rising_temperature_pandas(weather_df):
    """Solution 1: Using shift and date comparison"""
    weather_df = weather_df.sort_values('recordDate')
    weather_df['prev_temp'] = weather_df['temperature'].shift(1)
    weather_df['prev_date'] = weather_df['recordDate'].shift(1)

    result = weather_df[
        (weather_df['temperature'] > weather_df['prev_temp']) &
        (weather_df['recordDate'] == weather_df['prev_date'] + timedelta(days=1))
    ]
    return result[['id']]

def rising_temperature_merge(weather_df):
    """Solution 2: Using self merge"""
    weather_df['next_date'] = weather_df['recordDate'] + timedelta(days=1)

    result = weather_df.merge(
        weather_df,
        left_on='next_date',
        right_on='recordDate',
        suffixes=('_prev', '_curr')
    )
    result = result[result['temperature_curr'] > result['temperature_prev']]
    return result[['id_curr']].rename(columns={'id_curr': 'id'})

def rising_temperature_iterative(weather_list):
    """Solution 3: Using dictionary for O(n) lookup"""
    weather_dict = {w['recordDate']: w for w in weather_list}

    result = []
    for weather in weather_list:
        prev_date = weather['recordDate'] - timedelta(days=1)
        if prev_date in weather_dict:
            if weather['temperature'] > weather_dict[prev_date]['temperature']:
                result.append(weather['id'])
    return result

# ============================================================================
# 595. Big Countries
# ============================================================================

def big_countries_pandas(world_df):
    """Solution 1: Using boolean indexing"""
    result = world_df[
        (world_df['area'] >= 3000000) | (world_df['population'] >= 25000000)
    ]
    return result[['name', 'population', 'area']]

def big_countries_query(world_df):
    """Solution 2: Using DataFrame query"""
    result = world_df.query('area >= 3000000 or population >= 25000000')
    return result[['name', 'population', 'area']]

def big_countries_filter(world_list):
    """Solution 3: Using filter"""
    result = list(filter(
        lambda x: x['area'] >= 3000000 or x['population'] >= 25000000,
        world_list
    ))
    return [{'name': r['name'], 'population': r['population'], 'area': r['area']}
            for r in result]

# ============================================================================
# 607. Sales Person
# ============================================================================

def sales_person_pandas(salesperson_df, company_df, orders_df):
    """Solution 1: Using merge and isin"""
    red_orders = orders_df.merge(company_df, on='com_id')
    red_orders = red_orders[red_orders['name'] == 'RED']
    red_sales_ids = red_orders['sales_id'].unique()

    result = salesperson_df[~salesperson_df['sales_id'].isin(red_sales_ids)]
    return result[['name']]

def sales_person_set(salesperson_list, company_list, order_list):
    """Solution 2: Using set operations"""
    red_company_ids = {c['com_id'] for c in company_list if c['name'] == 'RED'}
    sales_to_red = {o['sales_id'] for o in order_list if o['com_id'] in red_company_ids}
    all_sales_ids = {sp['sales_id'] for sp in salesperson_list}

    valid_sales_ids = all_sales_ids - sales_to_red
    return [sp['name'] for sp in salesperson_list if sp['sales_id'] in valid_sales_ids]

def sales_person_nested(salesperson_df, company_df, orders_df):
    """Solution 3: Using nested conditions"""
    red_com_ids = company_df[company_df['name'] == 'RED']['com_id'].tolist()
    red_sales_ids = orders_df[orders_df['com_id'].isin(red_com_ids)]['sales_id'].unique()

    return salesperson_df[~salesperson_df['sales_id'].isin(red_sales_ids)][['name']]

# ============================================================================
# 620. Not Boring Movies
# ============================================================================

def not_boring_movies_pandas(cinema_df):
    """Solution 1: Using boolean indexing"""
    result = cinema_df[
        (cinema_df['id'] % 2 == 1) & (cinema_df['description'] != 'boring')
    ]
    return result.sort_values('rating', ascending=False)

def not_boring_movies_query(cinema_df):
    """Solution 2: Using DataFrame query"""
    result = cinema_df.query("id % 2 == 1 and description != 'boring'")
    return result.sort_values('rating', ascending=False)

def not_boring_movies_filter(cinema_list):
    """Solution 3: Using filter and sorted"""
    result = list(filter(
        lambda x: x['id'] % 2 == 1 and x['description'] != 'boring',
        cinema_list
    ))
    return sorted(result, key=lambda x: x['rating'], reverse=True)

# ============================================================================
# 1068. Product Sales Analysis I
# ============================================================================

def product_sales_pandas(sales_df, product_df):
    """Solution 1: Using merge"""
    result = sales_df.merge(product_df, on='product_id')
    return result[['product_name', 'year', 'price']]

def product_sales_join(sales_df, product_df):
    """Solution 2: Using join with set_index"""
    sales_with_name = sales_df.join(
        product_df.set_index('product_id')['product_name'],
        on='product_id'
    )
    return sales_with_name[['product_name', 'year', 'price']]

def product_sales_dict(sales_list, product_list):
    """Solution 3: Using dictionary lookup"""
    product_map = {p['product_id']: p['product_name'] for p in product_list}

    result = []
    for sale in sales_list:
        result.append({
            'product_name': product_map.get(sale['product_id']),
            'year': sale['year'],
            'price': sale['price']
        })
    return result

# ============================================================================
# 1322. Ads Performance
# ============================================================================

def ads_performance_pandas(ads_df):
    """Solution 1: Using groupby and apply"""
    def calculate_ctr(group):
        clicks = (group['action'] == 'Clicked').sum()
        views_and_clicks = group['action'].isin(['Clicked', 'Viewed']).sum()
        ctr = (clicks * 100.0 / views_and_clicks) if views_and_clicks > 0 else 0
        return round(ctr, 2)

    result = ads_df.groupby('ad_id').apply(calculate_ctr).reset_index()
    result.columns = ['ad_id', 'ctr']
    return result.sort_values(['ctr', 'ad_id'], ascending=[False, True])

def ads_performance_aggregate(ads_df):
    """Solution 2: Using aggregate functions"""
    filtered = ads_df[ads_df['action'].isin(['Clicked', 'Viewed'])]

    result = filtered.groupby('ad_id').agg(
        clicks=('action', lambda x: (x == 'Clicked').sum()),
        total=('action', 'count')
    ).reset_index()

    result['ctr'] = (result['clicks'] * 100.0 / result['total']).round(2)
    result['ctr'] = result['ctr'].fillna(0)

    return result[['ad_id', 'ctr']].sort_values(['ctr', 'ad_id'], ascending=[False, True])

def ads_performance_dict(ads_list):
    """Solution 3: Using dictionaries"""
    ad_stats = {}

    for ad in ads_list:
        ad_id = ad['ad_id']
        if ad_id not in ad_stats:
            ad_stats[ad_id] = {'clicks': 0, 'total': 0}

        if ad['action'] in ['Clicked', 'Viewed']:
            ad_stats[ad_id]['total'] += 1
            if ad['action'] == 'Clicked':
                ad_stats[ad_id]['clicks'] += 1

    result = []
    for ad_id, stats in ad_stats.items():
        ctr = round((stats['clicks'] * 100.0 / stats['total']), 2) if stats['total'] > 0 else 0
        result.append({'ad_id': ad_id, 'ctr': ctr})

    return

In [ ]:
provide multiple solutions using oracle sql,plsql,python -  177. Nth Highest Salary: Find the Nth highest salary.
    178. Rank Scores: Rank scores, handling ties appropriately.
    180. Consecutive Numbers: Find numbers that appear consecutively at least three times.
    184. Department Highest Salary: Find employees with the highest salary in each department.
    185. Department Top Three Salaries: Find employees with the top three salaries in each department.
    196. Delete Duplicate Emails: Delete duplicate email entries, keeping only one.
    601. Human Traffic of Stadium: Find periods of high human traffic in a stadium.
    626. Exchange Seats: Swap the seat IDs of adjacent students.
    1341. Movie Rating: Analyze movie ratings and user activity.
    1393. Capital Gain/Loss: Calculate the capital gain or loss for each stock.
    1454. Active Users: Identify users who were active for a certain number of consecutive days.
Hard:
    603. Consecutive Available Seats: Find three or more consecutive available seats in a cinema.
    1336. Number of Transactions per Visit: Calculate the number of transactions per visit.
    1369. Get the Second Most Recent Activity: Find the second most recent activity for each user.
    1412. Find the Quiet Students in All Exams: Identify students who are not the highest or lowest scorers in any exam.
--------
Select Recyclable and Low Fat Products
Find Customer Referee
Big Countries
Article Views I
Invalid Tweets
Replace Employee ID With The Unique Identifier
Product Sales Analysis I
Customer Who Visited but Did Not Make Any Transactions
Rising Temperature
Average Time of Process per Machine
Employee Bonus
Students and Examinations
Managers with at Least 5 Direct Reports
Confirmation Rate
Not Boring Movies
Average Selling Price
Project Employees I
Percentage of Users Attended a Contest
Queries Quality and Percentage
Monthly Transactions I
Immediate Food Delivery II
Game Play Analysis IV
Sorting and Grouping
Number of Unique Subjects Taught by Each Teacher
User Activity for the Past 30 Days I
Product Sales Analysis III
Classes With at Least 5 Students
Find Followers Count
Biggest Single Number
Customers Who Bought All Products
Advanced Select and Joins
The Number of Employees Which Report to Each Employee
Primary Department for Each Employee
Triangle Judgement
Consecutive Numbers
Product Price at a Given Date
Last Person to Fit in the Bus
Count Salary Categories
Employees Whose Manager Left the Company
Exchange Seats
Movie Rating
Restaurant Growth
Friend Requests II: Who Has the Most Friends
Investments in 2016
Department Top Three Salaries
Advanced String Functions / Regex / Clause -Fix Names in a Table
Patients With a Condition
Delete Duplicate Emails
Second Highest Salary
Group Sold Products By The Date
List the Products Ordered in a Period
Find Users With Valid E-Mails
---
Top Travellers
Trips and Users
Patients With a Condition
Fix Names in a Table
Reformat Department Table
Sales Analysis III
Game Play Analysis I
Rearrange Products Table
Daily Leads and Partners
Combine Two Tables
Second Highest Salary
Nth Highest Salary
Rank Scores
Consecutive Numbers
Employees Earning More Than Their Managers
Duplicate Emails
Customers Who Never Order
Department Highest Salary
Department Top Three Salaries
Customer Who Visited but Did Not Make Any Transactions
Employees With Missing Information
Delete Duplicate Emails
Rising Temperature
Bank Account Summary II
Big Countries
Classes With at Least 5 Students
Find Followers Count
Group Sold Products By The Date
User Activity for the Past 30 Days I
Sales Person
Tree Node
Find Total Time Spent by Each Employee
Calculate Special Bonus
Article Views I
Not Boring Movies
Actors and Directors Who Cooperated At Least Three Times
Exchange Seats
Swap Salary
Market Analysis I
Capital Gain/Loss
Recyclable and Low Fat Products
The Latest Login in 2020

In [ ]:
Comprehensive SQL Solutions for LeetCode Database Problems
I'll provide solutions for each problem using Oracle SQL, PL/SQL, and Python (with pandas).
========
-- ============================================================================
-- 177. Nth Highest Salary
-- ============================================================================

-- ORACLE SQL Solution 1: Using DENSE_RANK
CREATE FUNCTION getNthHighestSalary(N IN NUMBER) RETURN NUMBER IS
result NUMBER;
BEGIN
    SELECT DISTINCT salary INTO result
    FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) as rnk
        FROM Employee
    )
    WHERE rnk = N;
    RETURN result;
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RETURN NULL;
END;

-- ORACLE SQL Solution 2: Using OFFSET
CREATE FUNCTION getNthHighestSalary(N IN NUMBER) RETURN NUMBER IS
result NUMBER;
BEGIN
    SELECT DISTINCT salary INTO result
    FROM Employee
    ORDER BY salary DESC
    OFFSET N-1 ROWS FETCH NEXT 1 ROW ONLY;
    RETURN result;
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RETURN NULL;
END;

-- PL/SQL Solution: With Error Handling
CREATE OR REPLACE FUNCTION getNthHighestSalary(N IN NUMBER) RETURN NUMBER IS
    v_salary NUMBER;
    v_count NUMBER;
BEGIN
    IF N <= 0 THEN
        RETURN NULL;
    END IF;

    SELECT COUNT(DISTINCT salary) INTO v_count FROM Employee;

    IF N > v_count THEN
        RETURN NULL;
    END IF;

    SELECT salary INTO v_salary
    FROM (
        SELECT DISTINCT salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) as rn
        FROM Employee
    )
    WHERE rn = N;

    RETURN v_salary;
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RETURN NULL;
    WHEN OTHERS THEN
        RETURN NULL;
END getNthHighestSalary;

-- ============================================================================
-- 178. Rank Scores
-- ============================================================================

-- ORACLE SQL Solution: Using DENSE_RANK
SELECT score,
       DENSE_RANK() OVER (ORDER BY score DESC) as rank
FROM Scores
ORDER BY score DESC;

-- Alternative with RANK (allows gaps)
SELECT score,
       RANK() OVER (ORDER BY score DESC) as rank
FROM Scores
ORDER BY score DESC;

-- PL/SQL Procedure to populate ranked results
CREATE OR REPLACE PROCEDURE rank_scores IS
BEGIN
    MERGE INTO Scores_Ranked sr
    USING (
        SELECT id, score,
               DENSE_RANK() OVER (ORDER BY score DESC) as rank
        FROM Scores
    ) s
    ON (sr.id = s.id)
    WHEN MATCHED THEN
        UPDATE SET sr.rank = s.rank, sr.score = s.score
    WHEN NOT MATCHED THEN
        INSERT (id, score, rank)
        VALUES (s.id, s.score, s.rank);
    COMMIT;
END rank_scores;

-- ============================================================================
-- 180. Consecutive Numbers
-- ============================================================================

-- ORACLE SQL Solution 1: Using LAG/LEAD
SELECT DISTINCT num as ConsecutiveNums
FROM (
    SELECT num,
           LAG(num, 1) OVER (ORDER BY id) as prev_num,
           LAG(num, 2) OVER (ORDER BY id) as prev_prev_num
    FROM Logs
)
WHERE num = prev_num AND num = prev_prev_num;

-- ORACLE SQL Solution 2: Using Self-Join
SELECT DISTINCT l1.num as ConsecutiveNums
FROM Logs l1
JOIN Logs l2 ON l2.id = l1.id + 1 AND l2.num = l1.num
JOIN Logs l3 ON l3.id = l1.id + 2 AND l3.num = l1.num;

-- PL/SQL Solution with Cursor
CREATE OR REPLACE PROCEDURE find_consecutive_numbers IS
    CURSOR c_logs IS
        SELECT id, num
        FROM Logs
        ORDER BY id;

    TYPE t_nums IS TABLE OF NUMBER INDEX BY PLS_INTEGER;
    v_nums t_nums;
    v_idx PLS_INTEGER := 1;
BEGIN
    -- Load data into collection
    FOR rec IN c_logs LOOP
        v_nums(v_idx) := rec.num;
        v_idx := v_idx + 1;
    END LOOP;

    -- Find consecutive numbers
    FOR i IN 1..v_nums.COUNT - 2 LOOP
        IF v_nums(i) = v_nums(i+1) AND v_nums(i) = v_nums(i+2) THEN
            DBMS_OUTPUT.PUT_LINE('Consecutive Number: ' || v_nums(i));
        END IF;
    END LOOP;
END find_consecutive_numbers;

-- ============================================================================
-- 184. Department Highest Salary
-- ============================================================================

-- ORACLE SQL Solution: Using Window Function
SELECT d.name as Department,
       e.name as Employee,
       e.salary as Salary
FROM Employee e
JOIN Department d ON e.departmentId = d.id
WHERE (e.departmentId, e.salary) IN (
    SELECT departmentId, MAX(salary)
    FROM Employee
    GROUP BY departmentId
);

-- Alternative using RANK
SELECT Department, Employee, Salary
FROM (
    SELECT d.name as Department,
           e.name as Employee,
           e.salary as Salary,
           RANK() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC) as rnk
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
)
WHERE rnk = 1;

-- PL/SQL Solution
CREATE OR REPLACE PROCEDURE get_dept_highest_salaries IS
    CURSOR c_results IS
        SELECT d.name as department_name,
               e.name as employee_name,
               e.salary
        FROM Employee e
        JOIN Department d ON e.departmentId = d.id
        WHERE (e.departmentId, e.salary) IN (
            SELECT departmentId, MAX(salary)
            FROM Employee
            GROUP BY departmentId
        )
        ORDER BY d.name;
BEGIN
    FOR rec IN c_results LOOP
        DBMS_OUTPUT.PUT_LINE(
            'Dept: ' || rec.department_name ||
            ', Employee: ' || rec.employee_name ||
            ', Salary: ' || rec.salary
        );
    END LOOP;
END get_dept_highest_salaries;

-- ============================================================================
-- 185. Department Top Three Salaries
-- ============================================================================

-- ORACLE SQL Solution: Using DENSE_RANK
SELECT Department, Employee, Salary
FROM (
    SELECT d.name as Department,
           e.name as Employee,
           e.salary as Salary,
           DENSE_RANK() OVER (PARTITION BY d.id ORDER BY e.salary DESC) as rnk
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
)
WHERE rnk <= 3;

-- Alternative with ROW_NUMBER (no ties)
SELECT Department, Employee, Salary
FROM (
    SELECT d.name as Department,
           e.name as Employee,
           e.salary as Salary,
           ROW_NUMBER() OVER (PARTITION BY d.id ORDER BY e.salary DESC) as rn
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
)
WHERE rn <= 3;

-- PL/SQL Solution with Collections
CREATE OR REPLACE PROCEDURE get_top_three_salaries IS
    TYPE t_result IS RECORD (
        department VARCHAR2(100),
        employee VARCHAR2(100),
        salary NUMBER
    );
    TYPE t_results IS TABLE OF t_result;
    v_results t_results;
BEGIN
    SELECT Department, Employee, Salary
    BULK COLLECT INTO v_results
    FROM (
        SELECT d.name as Department,
               e.name as Employee,
               e.salary as Salary,
               DENSE_RANK() OVER (PARTITION BY d.id ORDER BY e.salary DESC) as rnk
        FROM Employee e
        JOIN Department d ON e.departmentId = d.id
    )
    WHERE rnk <= 3;

    FOR i IN 1..v_results.COUNT LOOP
        DBMS_OUTPUT.PUT_LINE(
            v_results(i).department || ' - ' ||
            v_results(i).employee || ': ' ||
            v_results(i).salary
        );
    END LOOP;
END get_top_three_salaries;

-- ============================================================================
-- 196. Delete Duplicate Emails
-- ============================================================================

-- ORACLE SQL Solution 1: Using ROWID (most efficient)
DELETE FROM Person
WHERE ROWID NOT IN (
    SELECT MIN(ROWID)
    FROM Person
    GROUP BY email
);

-- ORACLE SQL Solution 2: Using subquery with MIN(id)
DELETE FROM Person
WHERE id NOT IN (
    SELECT MIN(id)
    FROM Person
    GROUP BY email
);

-- ORACLE SQL Solution 3: Using self-join
DELETE FROM Person p1
WHERE EXISTS (
    SELECT 1
    FROM Person p2
    WHERE p1.email = p2.email
    AND p1.id > p2.id
);

-- PL/SQL Solution with explicit loop
CREATE OR REPLACE PROCEDURE delete_duplicate_emails IS
    CURSOR c_duplicates IS
        SELECT id
        FROM Person
        WHERE id NOT IN (
            SELECT MIN(id)
            FROM Person
            GROUP BY email
        );

    v_deleted_count NUMBER := 0;
BEGIN
    FOR rec IN c_duplicates LOOP
        DELETE FROM Person WHERE id = rec.id;
        v_deleted_count := v_deleted_count + 1;
    END LOOP;

    COMMIT;
    DBMS_OUTPUT.PUT_LINE('Deleted ' || v_deleted_count || ' duplicate emails');
EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        RAISE;
END delete_duplicate_emails;

-- Merge approach (keeping minimum ID)
MERGE INTO Person p1
USING (
    SELECT email, MIN(id) as min_id
    FROM Person
    GROUP BY email
    HAVING COUNT(*) > 1
) p2
ON (p1.email = p2.email AND p1.id > p2.min_id)
WHEN MATCHED THEN
    UPDATE SET p1.email = NULL
    DELETE WHERE p1.email IS NULL;

In [ ]:
-- ============================================================================
-- 601. Human Traffic of Stadium
-- ============================================================================

-- ORACLE SQL Solution: Find 3+ consecutive days with 100+ people
WITH numbered_rows AS (
    SELECT id, visit_date, people,
           id - ROW_NUMBER() OVER (ORDER BY id) as grp
    FROM Stadium
    WHERE people >= 100
),
consecutive_groups AS (
    SELECT grp, COUNT(*) as cnt,
           MIN(id) as min_id, MAX(id) as max_id
    FROM numbered_rows
    GROUP BY grp
    HAVING COUNT(*) >= 3
)
SELECT s.id, s.visit_date, s.people
FROM Stadium s
JOIN consecutive_groups cg
    ON s.id BETWEEN cg.min_id AND cg.max_id
ORDER BY s.visit_date;

-- Alternative using LAG/LEAD
SELECT DISTINCT s1.id, s1.visit_date, s1.people
FROM Stadium s1
WHERE s1.people >= 100
AND (
    (EXISTS (SELECT 1 FROM Stadium s2 WHERE s2.id = s1.id - 1 AND s2.people >= 100)
     AND EXISTS (SELECT 1 FROM Stadium s3 WHERE s3.id = s1.id - 2 AND s3.people >= 100))
    OR
    (EXISTS (SELECT 1 FROM Stadium s2 WHERE s2.id = s1.id - 1 AND s2.people >= 100)
     AND EXISTS (SELECT 1 FROM Stadium s3 WHERE s3.id = s1.id + 1 AND s3.people >= 100))
    OR
    (EXISTS (SELECT 1 FROM Stadium s2 WHERE s2.id = s1.id + 1 AND s2.people >= 100)
     AND EXISTS (SELECT 1 FROM Stadium s3 WHERE s3.id = s1.id + 2 AND s3.people >= 100))
)
ORDER BY s1.visit_date;

-- PL/SQL Solution
CREATE OR REPLACE PROCEDURE find_high_traffic_periods IS
    CURSOR c_stadium IS
        SELECT id, visit_date, people
        FROM Stadium
        ORDER BY id;

    TYPE t_record IS RECORD (
        id NUMBER,
        visit_date DATE,
        people NUMBER
    );
    TYPE t_records IS TABLE OF t_record;
    v_records t_records := t_records();
    v_consecutive NUMBER := 0;
    v_start_idx NUMBER;
BEGIN
    FOR rec IN c_stadium LOOP
        v_records.EXTEND;
        v_records(v_records.LAST) := rec;
    END LOOP;

    FOR i IN 1..v_records.COUNT LOOP
        IF v_records(i).people >= 100 THEN
            v_consecutive := 1;
            v_start_idx := i;

            FOR j IN i+1..LEAST(i+2, v_records.COUNT) LOOP
                IF v_records(j).id = v_records(j-1).id + 1
                   AND v_records(j).people >= 100 THEN
                    v_consecutive := v_consecutive + 1;
                ELSE
                    EXIT;
                END IF;
            END LOOP;

            IF v_consecutive >= 3 THEN
                FOR k IN v_start_idx..v_start_idx + v_consecutive - 1 LOOP
                    DBMS_OUTPUT.PUT_LINE(
                        'ID: ' || v_records(k).id ||
                        ', Date: ' || v_records(k).visit_date ||
                        ', People: ' || v_records(k).people
                    );
                END LOOP;
            END IF;
        END IF;
    END LOOP;
END find_high_traffic_periods;

-- ============================================================================
-- 626. Exchange Seats
-- ============================================================================

-- ORACLE SQL Solution 1: Using CASE
SELECT
    CASE
        WHEN MOD(id, 2) = 1 AND id != (SELECT MAX(id) FROM Seat) THEN id + 1
        WHEN MOD(id, 2) = 0 THEN id - 1
        ELSE id
    END as id,
    student
FROM Seat
ORDER BY id;

-- ORACLE SQL Solution 2: Using LEAD/LAG
SELECT
    CASE
        WHEN MOD(id, 2) = 1 THEN COALESCE(LEAD(id) OVER (ORDER BY id), id)
        ELSE LAG(id) OVER (ORDER BY id)
    END as id,
    student
FROM Seat
ORDER BY id;

-- PL/SQL Solution
CREATE OR REPLACE PROCEDURE exchange_seats IS
    TYPE t_seat IS RECORD (
        id NUMBER,
        student VARCHAR2(100)
    );
    TYPE t_seats IS TABLE OF t_seat;
    v_seats t_seats;
    v_temp_student VARCHAR2(100);
BEGIN
    SELECT id, student
    BULK COLLECT INTO v_seats
    FROM Seat
    ORDER BY id;

    FOR i IN 1..v_seats.COUNT LOOP
        IF MOD(i, 2) = 1 AND i < v_seats.COUNT THEN
            -- Swap with next
            v_temp_student := v_seats(i).student;
            v_seats(i).student := v_seats(i+1).student;
            v_seats(i+1).student := v_temp_student;
        END IF;
    END LOOP;

    FOR i IN 1..v_seats.COUNT LOOP
        DBMS_OUTPUT.PUT_LINE('ID: ' || i || ', Student: ' || v_seats(i).student);
    END LOOP;
END exchange_seats;

-- ============================================================================
-- 1341. Movie Rating
-- ============================================================================

-- ORACLE SQL Solution: Two part query
(
    SELECT name as results
    FROM (
        SELECT u.name, COUNT(*) as rating_count
        FROM MovieRating mr
        JOIN Users u ON mr.user_id = u.user_id
        GROUP BY u.name
        ORDER BY rating_count DESC, u.name
    )
    WHERE ROWNUM = 1
)
UNION ALL
(
    SELECT title as results
    FROM (
        SELECT m.title, AVG(mr.rating) as avg_rating
        FROM MovieRating mr
        JOIN Movies m ON mr.movie_id = m.movie_id
        WHERE TO_CHAR(mr.created_at, 'YYYY-MM') = '2020-02'
        GROUP BY m.title
        ORDER BY avg_rating DESC, m.title
    )
    WHERE ROWNUM = 1
);

-- Using FETCH FIRST (Oracle 12c+)
SELECT results FROM (
    SELECT name as results, 1 as ord
    FROM (
        SELECT u.name, COUNT(*) as cnt
        FROM MovieRating mr
        JOIN Users u ON mr.user_id = u.user_id
        GROUP BY u.name
        ORDER BY cnt DESC, u.name
        FETCH FIRST 1 ROW ONLY
    )
    UNION ALL
    SELECT title as results, 2 as ord
    FROM (
        SELECT m.title, AVG(mr.rating) as avg_rating
        FROM MovieRating mr
        JOIN Movies m ON mr.movie_id = m.movie_id
        WHERE TRUNC(mr.created_at, 'MM') = DATE '2020-02-01'
        GROUP BY m.title
        ORDER BY avg_rating DESC, m.title
        FETCH FIRST 1 ROW ONLY
    )
)
ORDER BY ord;

-- ============================================================================
-- 1393. Capital Gain/Loss
-- ============================================================================

-- ORACLE SQL Solution
SELECT stock_name,
       SUM(CASE WHEN operation = 'Buy' THEN -price ELSE price END) as capital_gain_loss
FROM Stocks
GROUP BY stock_name;

-- Alternative approach
SELECT stock_name,
       SUM(sell_price) - SUM(buy_price) as capital_gain_loss
FROM (
    SELECT stock_name,
           CASE WHEN operation = 'Sell' THEN price ELSE 0 END as sell_price,
           CASE WHEN operation = 'Buy' THEN price ELSE 0 END as buy_price
    FROM Stocks
)
GROUP BY stock_name;

-- PL/SQL Solution
CREATE OR REPLACE PROCEDURE calculate_capital_gain IS
    CURSOR c_stocks IS
        SELECT stock_name,
               SUM(CASE WHEN operation = 'Buy' THEN -price ELSE price END) as gain_loss
        FROM Stocks
        GROUP BY stock_name;
BEGIN
    FOR rec IN c_stocks LOOP
        DBMS_OUTPUT.PUT_LINE(
            'Stock: ' || rec.stock_name ||
            ', Gain/Loss: ' || rec.gain_loss
        );
    END LOOP;
END calculate_capital_gain;

-- ============================================================================
-- 1454. Active Users
-- ============================================================================

-- ORACLE SQL Solution: Find users active 5+ consecutive days
WITH user_dates AS (
    SELECT DISTINCT user_id, activity_date
    FROM Logins
),
numbered AS (
    SELECT user_id, activity_date,
           activity_date - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY activity_date) as grp
    FROM user_dates
),
consecutive_days AS (
    SELECT user_id, COUNT(*) as consecutive_count
    FROM numbered
    GROUP BY user_id, grp
    HAVING COUNT(*) >= 5
)
SELECT DISTINCT u.id, u.name
FROM Users u
JOIN consecutive_days cd ON u.id = cd.user_id
ORDER BY u.id;

-- Alternative using window functions
SELECT DISTINCT a.id, a.name
FROM Accounts a
WHERE a.id IN (
    SELECT user_id
    FROM (
        SELECT user_id,
               login_date,
               login_date - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date) as date_group
        FROM (
            SELECT DISTINCT user_id, login_date
            FROM Logins
        )
    )
    GROUP BY user_id, date_group
    HAVING COUNT(DISTINCT login_date) >= 5
)
ORDER BY a.id;

-- PL/SQL Solution
CREATE OR REPLACE PROCEDURE find_active_users(p_consecutive_days IN NUMBER DEFAULT 5) IS
    CURSOR c_users IS
        SELECT DISTINCT user_id
        FROM (
            SELECT user_id,
                   activity_date - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY activity_date) as grp
            FROM (SELECT DISTINCT user_id, activity_date FROM Logins)
        )
        GROUP BY user_id, grp
        HAVING COUNT(*) >= p_consecutive_days;

    v_user_name VARCHAR2(100);
BEGIN
    FOR rec IN c_users LOOP
        SELECT name INTO v_user_name
        FROM Users
        WHERE id = rec.user_id;

        DBMS_OUTPUT.PUT_LINE('Active User: ' || rec.user_id || ' - ' || v_user_name);
    END LOOP;
END find_active_users;

In [ ]:
-- ============================================================================
-- 603. Consecutive Available Seats (HARD)
-- ============================================================================

-- ORACLE SQL Solution 1: Using self-join
SELECT DISTINCT c1.seat_id
FROM Cinema c1
JOIN Cinema c2
    ON ABS(c1.seat_id - c2.seat_id) = 1
    AND c1.free = 1
    AND c2.free = 1
ORDER BY c1.seat_id;

-- ORACLE SQL Solution 2: Using LAG/LEAD
SELECT seat_id
FROM (
    SELECT seat_id,
           free,
           LAG(free) OVER (ORDER BY seat_id) as prev_free,
           LEAD(free) OVER (ORDER BY seat_id) as next_free
    FROM Cinema
)
WHERE free = 1
  AND (prev_free = 1 OR next_free = 1)
ORDER BY seat_id;

-- PL/SQL Solution: Find groups of 3+ consecutive seats
CREATE OR REPLACE PROCEDURE find_consecutive_seats IS
    TYPE t_seat IS RECORD (
        seat_id NUMBER,
        free NUMBER
    );
    TYPE t_seats IS TABLE OF t_seat;
    v_seats t_seats;
    v_consecutive NUMBER := 0;
    v_start_idx NUMBER;
BEGIN
    SELECT seat_id, free
    BULK COLLECT INTO v_seats
    FROM Cinema
    ORDER BY seat_id;

    FOR i IN 1..v_seats.COUNT LOOP
        IF v_seats(i).free = 1 THEN
            IF v_consecutive = 0 THEN
                v_start_idx := i;
            END IF;
            v_consecutive := v_consecutive + 1;
        ELSE
            IF v_consecutive >= 3 THEN
                DBMS_OUTPUT.PUT_LINE('Found ' || v_consecutive ||
                    ' consecutive seats starting at seat ' || v_seats(v_start_idx).seat_id);
            END IF;
            v_consecutive := 0;
        END IF;
    END LOOP;

    -- Check last sequence
    IF v_consecutive >= 3 THEN
        DBMS_OUTPUT.PUT_LINE('Found ' || v_consecutive ||
            ' consecutive seats starting at seat ' || v_seats(v_start_idx).seat_id);
    END IF;
END find_consecutive_seats;

-- ============================================================================
-- 1336. Number of Transactions per Visit (HARD)
-- ============================================================================

-- ORACLE SQL Solution: Complex aggregation
WITH RECURSIVE numbers AS (
    SELECT 0 as n
    UNION ALL
    SELECT n + 1
    FROM numbers
    WHERE n < (SELECT MAX(transaction_count) FROM (
        SELECT user_id, visit_date, COUNT(transaction_date) as transaction_count
        FROM Visits v
        LEFT JOIN Transactions t
            ON v.user_id = t.user_id
            AND v.visit_date = t.transaction_date
        GROUP BY v.user_id, v.visit_date
    ))
),
visit_transactions AS (
    SELECT v.user_id, v.visit_date,
           COUNT(t.transaction_date) as transaction_count
    FROM Visits v
    LEFT JOIN Transactions t
        ON v.user_id = t.user_id
        AND v.visit_date = t.transaction_date
    GROUP BY v.user_id, v.visit_date
)
SELECT n.n as transactions_count,
       NVL(COUNT(vt.transaction_count), 0) as visits_count
FROM numbers n
LEFT JOIN visit_transactions vt ON n.n = vt.transaction_count
GROUP BY n.n
ORDER BY n.n;

-- Oracle version without RECURSIVE (using LEVEL)
WITH transaction_counts AS (
    SELECT COALESCE(COUNT(t.transaction_date), 0) as trans_count
    FROM Visits v
    LEFT JOIN Transactions t
        ON v.user_id = t.user_id
        AND v.visit_date = t.transaction_date
    GROUP BY v.user_id, v.visit_date
),
max_transactions AS (
    SELECT NVL(MAX(trans_count), 0) as max_trans
    FROM transaction_counts
)
SELECT LEVEL - 1 as transactions_count,
       NVL(SUM(CASE WHEN tc.trans_count = LEVEL - 1 THEN 1 ELSE 0 END), 0) as visits_count
FROM max_transactions mt
CROSS JOIN transaction_counts tc
WHERE LEVEL - 1 <= mt.max_trans
CONNECT BY LEVEL <= (SELECT max_trans + 1 FROM max_transactions)
GROUP BY LEVEL - 1
ORDER BY transactions_count;

-- ============================================================================
-- 1369. Get the Second Most Recent Activity (HARD)
-- ============================================================================

-- ORACLE SQL Solution
WITH ranked_activities AS (
    SELECT username, activity, startDate, endDate,
           ROW_NUMBER() OVER (PARTITION BY username ORDER BY startDate DESC) as rn,
           COUNT(*) OVER (PARTITION BY username) as total_activities
    FROM UserActivity
)
SELECT username, activity, startDate, endDate
FROM ranked_activities
WHERE rn = 2
   OR (total_activities = 1 AND rn = 1)
ORDER BY username;

-- Alternative using RANK
WITH activity_ranks AS (
    SELECT username, activity, startDate, endDate,
           RANK() OVER (PARTITION BY username ORDER BY startDate DESC) as rnk,
           COUNT(*) OVER (PARTITION BY username) as cnt
    FROM UserActivity
)
SELECT username, activity, startDate, endDate
FROM activity_ranks
WHERE rnk = 2 OR (cnt = 1 AND rnk = 1);

-- PL/SQL Solution
CREATE OR REPLACE PROCEDURE get_second_recent_activity IS
    CURSOR c_activities IS
        WITH ranked AS (
            SELECT username, activity, startDate, endDate,
                   ROW_NUMBER() OVER (PARTITION BY username ORDER BY startDate DESC) as rn,
                   COUNT(*) OVER (PARTITION BY username) as cnt
            FROM UserActivity
        )
        SELECT username, activity, startDate, endDate
        FROM ranked
        WHERE rn = 2 OR (cnt = 1 AND rn = 1)
        ORDER BY username;
BEGIN
    FOR rec IN c_activities LOOP
        DBMS_OUTPUT.PUT_LINE(
            'User: ' || rec.username ||
            ', Activity: ' || rec.activity ||
            ', Start: ' || rec.startDate
        );
    END LOOP;
END get_second_recent_activity;

-- ============================================================================
-- 1412. Find the Quiet Students in All Exams (HARD)
-- ============================================================================

-- ORACLE SQL Solution
WITH exam_ranks AS (
    SELECT exam_id, student_id, score,
           RANK() OVER (PARTITION BY exam_id ORDER BY score DESC) as high_rank,
           RANK() OVER (PARTITION BY exam_id ORDER BY score ASC) as low_rank
    FROM Exam
),
noisy_students AS (
    SELECT DISTINCT student_id
    FROM exam_ranks
    WHERE high_rank = 1 OR low_rank = 1
)
SELECT DISTINCT s.student_id, s.student_name
FROM Student s
JOIN Exam e ON s.student_id = e.student_id
WHERE s.student_id NOT IN (SELECT student_id FROM noisy_students)
ORDER BY s.student_id;

-- Alternative approach with NOT EXISTS
SELECT DISTINCT s.student_id, s.student_name
FROM Student s
WHERE EXISTS (
    SELECT 1 FROM Exam e WHERE e.student_id = s.student_id
)
AND NOT EXISTS (
    SELECT 1
    FROM Exam e1
    WHERE e1.student_id = s.student_id
    AND (
        e1.score = (SELECT MAX(score) FROM Exam e2 WHERE e2.exam_id = e1.exam_id)
        OR
        e1.score = (SELECT MIN(score) FROM Exam e2 WHERE e2.exam_id = e1.exam_id)
    )
)
ORDER BY s.student_id;

-- PL/SQL Solution
CREATE OR REPLACE PROCEDURE find_quiet_students IS
    CURSOR c_quiet IS
        WITH exam_extremes AS (
            SELECT exam_id,
                   MAX(score) as max_score,
                   MIN(score) as min_score
            FROM Exam
            GROUP BY exam_id
        ),
        noisy AS (
            SELECT DISTINCT e.student_id
            FROM Exam e
            JOIN exam_extremes ee ON e.exam_id = ee.exam_id
            WHERE e.score = ee.max_score OR e.score = ee.min_score
        )
        SELECT DISTINCT s.student_id, s.student_name
        FROM Student s
        JOIN Exam e ON s.student_id = e.student_id
        WHERE s.student_id NOT IN (SELECT student_id FROM noisy)
        ORDER BY s.student_id;
BEGIN
    FOR rec IN c_quiet LOOP
        DBMS_OUTPUT.PUT_LINE(

In [ ]:
-- ============================================================================
-- 603. Consecutive Available Seats (HARD)
-- ============================================================================

-- ORACLE SQL Solution 1: Using self-join
SELECT DISTINCT c1.seat_id
FROM Cinema c1
JOIN Cinema c2
    ON ABS(c1.seat_id - c2.seat_id) = 1
    AND c1.free = 1
    AND c2.free = 1
ORDER BY c1.seat_id;

-- ORACLE SQL Solution 2: Using LAG/LEAD
SELECT seat_id
FROM (
    SELECT seat_id,
           free,
           LAG(free) OVER (ORDER BY seat_id) as prev_free,
           LEAD(free) OVER (ORDER BY seat_id) as next_free
    FROM Cinema
)
WHERE free = 1
  AND (prev_free = 1 OR next_free = 1)
ORDER BY seat_id;

-- PL/SQL Solution: Find groups of 3+ consecutive seats
CREATE OR REPLACE PROCEDURE find_consecutive_seats IS
    TYPE t_seat IS RECORD (
        seat_id NUMBER,
        free NUMBER
    );
    TYPE t_seats IS TABLE OF t_seat;
    v_seats t_seats;
    v_consecutive NUMBER := 0;
    v_start_idx NUMBER;
BEGIN
    SELECT seat_id, free
    BULK COLLECT INTO v_seats
    FROM Cinema
    ORDER BY seat_id;

    FOR i IN 1..v_seats.COUNT LOOP
        IF v_seats(i).free = 1 THEN
            IF v_consecutive = 0 THEN
                v_start_idx := i;
            END IF;
            v_consecutive := v_consecutive + 1;
        ELSE
            IF v_consecutive >= 3 THEN
                DBMS_OUTPUT.PUT_LINE('Found ' || v_consecutive ||
                    ' consecutive seats starting at seat ' || v_seats(v_start_idx).seat_id);
            END IF;
            v_consecutive := 0;
        END IF;
    END LOOP;

    -- Check last sequence
    IF v_consecutive >= 3 THEN
        DBMS_OUTPUT.PUT_LINE('Found ' || v_consecutive ||
            ' consecutive seats starting at seat ' || v_seats(v_start_idx).seat_id);
    END IF;
END find_consecutive_seats;

-- ============================================================================
-- 1336. Number of Transactions per Visit (HARD)
-- ============================================================================

-- ORACLE SQL Solution: Complex aggregation
WITH RECURSIVE numbers AS (
    SELECT 0 as n
    UNION ALL
    SELECT n + 1
    FROM numbers
    WHERE n < (SELECT MAX(transaction_count) FROM (
        SELECT user_id, visit_date, COUNT(transaction_date) as transaction_count
        FROM Visits v
        LEFT JOIN Transactions t
            ON v.user_id = t.user_id
            AND v.visit_date = t.transaction_date
        GROUP BY v.user_id, v.visit_date
    ))
),
visit_transactions AS (
    SELECT v.user_id, v.visit_date,
           COUNT(t.transaction_date) as transaction_count
    FROM Visits v
    LEFT JOIN Transactions t
        ON v.user_id = t.user_id
        AND v.visit_date = t.transaction_date
    GROUP BY v.user_id, v.visit_date
)
SELECT n.n as transactions_count,
       NVL(COUNT(vt.transaction_count), 0) as visits_count
FROM numbers n
LEFT JOIN visit_transactions vt ON n.n = vt.transaction_count
GROUP BY n.n
ORDER BY n.n;

-- Oracle version without RECURSIVE (using LEVEL)
WITH transaction_counts AS (
    SELECT COALESCE(COUNT(t.transaction_date), 0) as trans_count
    FROM Visits v
    LEFT JOIN Transactions t
        ON v.user_id = t.user_id
        AND v.visit_date = t.transaction_date
    GROUP BY v.user_id, v.visit_date
),
max_transactions AS (
    SELECT NVL(MAX(trans_count), 0) as max_trans
    FROM transaction_counts
)
SELECT LEVEL - 1 as transactions_count,
       NVL(SUM(CASE WHEN tc.trans_count = LEVEL - 1 THEN 1 ELSE 0 END), 0) as visits_count
FROM max_transactions mt
CROSS JOIN transaction_counts tc
WHERE LEVEL - 1 <= mt.max_trans
CONNECT BY LEVEL <= (SELECT max_trans + 1 FROM max_transactions)
GROUP BY LEVEL - 1
ORDER BY transactions_count;

-- ============================================================================
-- 1369. Get the Second Most Recent Activity (HARD)
-- ============================================================================

-- ORACLE SQL Solution
WITH ranked_activities AS (
    SELECT username, activity, startDate, endDate,
           ROW_NUMBER() OVER (PARTITION BY username ORDER BY startDate DESC) as rn,
           COUNT(*) OVER (PARTITION BY username) as total_activities
    FROM UserActivity
)
SELECT username, activity, startDate, endDate
FROM ranked_activities
WHERE rn = 2
   OR (total_activities = 1 AND rn = 1)
ORDER BY username;

-- Alternative using RANK
WITH activity_ranks AS (
    SELECT username, activity, startDate, endDate,
           RANK() OVER (PARTITION BY username ORDER BY startDate DESC) as rnk,
           COUNT(*) OVER (PARTITION BY username) as cnt
    FROM UserActivity
)
SELECT username, activity, startDate, endDate
FROM activity_ranks
WHERE rnk = 2 OR (cnt = 1 AND rnk = 1);

-- PL/SQL Solution
CREATE OR REPLACE PROCEDURE get_second_recent_activity IS
    CURSOR c_activities IS
        WITH ranked AS (
            SELECT username, activity, startDate, endDate,
                   ROW_NUMBER() OVER (PARTITION BY username ORDER BY startDate DESC) as rn,
                   COUNT(*) OVER (PARTITION BY username) as cnt
            FROM UserActivity
        )
        SELECT username, activity, startDate, endDate
        FROM ranked
        WHERE rn = 2 OR (cnt = 1 AND rn = 1)
        ORDER BY username;
BEGIN
    FOR rec IN c_activities LOOP
        DBMS_OUTPUT.PUT_LINE(
            'User: ' || rec.username ||
            ', Activity: ' || rec.activity ||
            ', Start: ' || rec.startDate
        );
    END LOOP;
END get_second_recent_activity;

-- ============================================================================
-- 1412. Find the Quiet Students in All Exams (HARD)
-- ============================================================================

-- ORACLE SQL Solution
WITH exam_ranks AS (
    SELECT exam_id, student_id, score,
           RANK() OVER (PARTITION BY exam_id ORDER BY score DESC) as high_rank,
           RANK() OVER (PARTITION BY exam_id ORDER BY score ASC) as low_rank
    FROM Exam
),
noisy_students AS (
    SELECT DISTINCT student_id
    FROM exam_ranks
    WHERE high_rank = 1 OR low_rank = 1
)
SELECT DISTINCT s.student_id, s.student_name
FROM Student s
JOIN Exam e ON s.student_id = e.student_id
WHERE s.student_id NOT IN (SELECT student_id FROM noisy_students)
ORDER BY s.student_id;

-- Alternative approach with NOT EXISTS
SELECT DISTINCT s.student_id, s.student_name
FROM Student s
WHERE EXISTS (
    SELECT 1 FROM Exam e WHERE e.student_id = s.student_id
)
AND NOT EXISTS (
    SELECT 1
    FROM Exam e1
    WHERE e1.student_id = s.student_id
    AND (
        e1.score = (SELECT MAX(score) FROM Exam e2 WHERE e2.exam_id = e1.exam_id)
        OR
        e1.score = (SELECT MIN(score) FROM Exam e2 WHERE e2.exam_id = e1.exam_id)
    )
)
ORDER BY s.student_id;

-- PL/SQL Solution
CREATE OR REPLACE PROCEDURE find_quiet_students IS
    CURSOR c_quiet IS
        WITH exam_extremes AS (
            SELECT exam_id,
                   MAX(score) as max_score,
                   MIN(score) as min_score
            FROM Exam
            GROUP BY exam_id
        ),
        noisy AS (
            SELECT DISTINCT e.student_id
            FROM Exam e
            JOIN exam_extremes ee ON e.exam_id = ee.exam_id
            WHERE e.score = ee.max_score OR e.score = ee.min_score
        )
        SELECT DISTINCT s.student_id, s.student_name
        FROM Student s
        JOIN Exam e ON s.student_id = e.student_id
        WHERE s.student_id NOT IN (SELECT student_id FROM noisy)
        ORDER BY s.student_id;
BEGIN
    FOR rec IN c_quiet LOOP
        DBMS_OUTPUT.PUT_LINE(
            'Quiet Student ID: ' || rec.student_id ||
            ', Name: ' || rec.student_name
        );
    END LOOP;
END find_quiet_students;

In [ ]:
import pandas as pd
import numpy as np

# ============================================================================
# 177. Nth Highest Salary
# ============================================================================

def nth_highest_salary(employee: pd.DataFrame, N: int) -> pd.DataFrame:
    """Find the Nth highest salary"""
    # Remove duplicates and sort
    unique_salaries = employee['salary'].drop_duplicates().sort_values(ascending=False)

    # Check if N is valid
    if N <= 0 or N > len(unique_salaries):
        return pd.DataFrame({f'getNthHighestSalary({N})': [None]})

    # Get Nth highest
    nth_salary = unique_salaries.iloc[N-1]
    return pd.DataFrame({f'getNthHighestSalary({N})': [nth_salary]})

# Alternative using rank
def nth_highest_salary_v2(employee: pd.DataFrame, N: int) -> pd.DataFrame:
    employee['rank'] = employee['salary'].rank(method='dense', ascending=False)
    result = employee[employee['rank'] == N]['salary'].drop_duplicates()

    if result.empty:
        return pd.DataFrame({f'getNthHighestSalary({N})': [None]})
    return pd.DataFrame({f'getNthHighestSalary({N})': [result.iloc[0]]})


# ============================================================================
# 178. Rank Scores
# ============================================================================

def rank_scores(scores: pd.DataFrame) -> pd.DataFrame:
    """Rank scores using dense ranking"""
    scores['rank'] = scores['score'].rank(method='dense', ascending=False)
    return scores[['score', 'rank']].sort_values('score', ascending=False)

# Alternative implementation
def rank_scores_v2(scores: pd.DataFrame) -> pd.DataFrame:
    result = scores.copy()
    result['rank'] = result['score'].rank(method='dense', ascending=False).astype(int)
    return result[['score', 'rank']].sort_values('rank')


# ============================================================================
# 180. Consecutive Numbers
# ============================================================================

def consecutive_numbers(logs: pd.DataFrame) -> pd.DataFrame:
    """Find numbers appearing at least 3 times consecutively"""
    logs = logs.sort_values('id')
    logs['prev1'] = logs['num'].shift(1)
    logs['prev2'] = logs['num'].shift(2)

    # Find rows where current, prev1, and prev2 are all the same
    consecutive = logs[
        (logs['num'] == logs['prev1']) &
        (logs['num'] == logs['prev2'])
    ]['num'].unique()

    return pd.DataFrame({'ConsecutiveNums': consecutive})

# Alternative using rolling window
def consecutive_numbers_v2(logs: pd.DataFrame) -> pd.DataFrame:
    logs = logs.sort_values('id').reset_index(drop=True)

    consecutive_nums = []
    for i in range(len(logs) - 2):
        if logs.loc[i, 'num'] == logs.loc[i+1, 'num'] == logs.loc[i+2, 'num']:
            consecutive_nums.append(logs.loc[i, 'num'])

    return pd.DataFrame({'ConsecutiveNums': list(set(consecutive_nums))})


# ============================================================================
# 184. Department Highest Salary
# ============================================================================

def department_highest_salary(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
    """Find employees with highest salary in each department"""
    # Merge employee with department
    merged = employee.merge(department, left_on='departmentId', right_on='id', suffixes=('_emp', '_dept'))

    # Find max salary per department
    max_salaries = merged.groupby('departmentId')['salary'].max().reset_index()
    max_salaries.columns = ['departmentId', 'max_salary']

    # Join to get employees with max salary
    result = merged.merge(max_salaries, on='departmentId')
    result = result[result['salary'] == result['max_salary']]

    return result[['name_dept', 'name_emp', 'salary']].rename(
        columns={'name_dept': 'Department', 'name_emp': 'Employee', 'salary': 'Salary'}
    )


# ============================================================================
# 185. Department Top Three Salaries
# ============================================================================

def top_three_salaries(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
    """Find top 3 salaries in each department"""
    # Merge tables
    merged = employee.merge(department, left_on='departmentId', right_on='id', suffixes=('_emp', '_dept'))

    # Rank salaries within each department
    merged['rank'] = merged.groupby('departmentId')['salary'].rank(method='dense', ascending=False)

    # Filter top 3
    result = merged[merged['rank'] <= 3]

    return result[['name_dept', 'name_emp', 'salary']].rename(
        columns={'name_dept': 'Department', 'name_emp': 'Employee', 'salary': 'Salary'}
    ).sort_values(['Department', 'salary'], ascending=[True, False])


# ============================================================================
# 196. Delete Duplicate Emails
# ============================================================================

def delete_duplicate_emails(person: pd.DataFrame) -> pd.DataFrame:
    """Delete duplicate emails, keeping the one with smallest id"""
    # Keep first occurrence (smallest id) for each email
    person.sort_values('id', inplace=True)
    person.drop_duplicates(subset='email', keep='first', inplace=True)
    return person

# Alternative approach
def delete_duplicate_emails_v2(person: pd.DataFrame) -> None:
    """In-place deletion of duplicates"""
    min_ids = person.groupby('email')['id'].min()
    person.drop(person[~person['id'].isin(min_ids)].index, inplace=True)


# ============================================================================
# 601. Human Traffic of Stadium
# ============================================================================

def human_traffic(stadium: pd.DataFrame) -> pd.DataFrame:
    """Find 3+ consecutive days with 100+ people"""
    stadium = stadium[stadium['people'] >= 100].sort_values('id').reset_index(drop=True)

    if len(stadium) < 3:
        return pd.DataFrame(columns=['id', 'visit_date', 'people'])

    # Create group identifier for consecutive IDs
    stadium['group'] = stadium['id'] - stadium.index

    # Count consecutive days in each group
    group_counts = stadium.groupby('group').size()
    valid_groups = group_counts[group_counts >= 3].index

    # Filter to valid groups
    result = stadium[stadium['group'].isin(valid_groups)]
    return result[['id', 'visit_date', 'people']].sort_values('visit_date')


# ============================================================================
# 626. Exchange Seats
# ============================================================================

def exchange_seats(seat: pd.DataFrame) -> pd.DataFrame:
    """Swap adjacent student seats"""
    result = seat.copy()
    max_id = result['id'].max()

    # Swap odd with next even (if exists)
    result['id'] = result.apply(
        lambda row: row['id'] + 1 if row['id'] % 2 == 1 and row['id'] != max_id
        else row['id'] - 1 if row['id'] % 2 == 0
        else row['id'],
        axis=1
    )

    return result.sort_values('id')

# Alternative using numpy
def exchange_seats_v2(seat: pd.DataFrame) -> pd.DataFrame:
    seat = seat.copy()
    n = len(seat)

    # Swap in pairs
    for i in range(0, n-1, 2):
        seat.loc[i, 'student'], seat.loc[i+1, 'student'] = \
            seat.loc[i+1, 'student'], seat.loc[i, 'student']

    return seat


# ============================================================================
# 1341. Movie Rating
# ============================================================================

def movie_rating(movies: pd.DataFrame, users: pd.DataFrame,
                movie_rating: pd.DataFrame) -> pd.DataFrame:
    """Find user with most ratings and highest avg rated movie in Feb 2020"""

    # Part 1: User with most ratings
    user_counts = movie_rating.merge(users, on='user_id')
    user_counts = user_counts.groupby('name').size().reset_index(name='count')
    user_counts = user_counts.sort_values(['count', 'name'], ascending=[False, True])
    top_user = user_counts.iloc[0]['name']

    # Part 2: Movie with highest average rating in Feb 2020
    feb_ratings = movie_rating[
        (movie_rating['created_at'] >= '2020-02-01') &
        (movie_rating['created_at'] < '2020-03-01')
    ]
    movie_avg = feb_ratings.merge(movies, on='movie_id')
    movie_avg = movie_avg.groupby('title')['rating'].mean().reset_index()
    movie_avg = movie_avg.sort_values(['rating', 'title'], ascending=[False, True])
    top_movie = movie_avg.iloc[0]['title']

    return pd.DataFrame({'results': [top_user, top_movie]})


# ============================================================================
# 1393. Capital Gain/Loss
# ============================================================================

def capital_gainloss(stocks: pd.DataFrame) -> pd.DataFrame:
    """Calculate capital gain/loss for each stock"""
    stocks['amount'] = stocks.apply(
        lambda row: -row['price'] if row['operation'] == 'Buy' else row['price'],
        axis=1
    )

    result = stocks.groupby('stock_name')['amount'].sum().reset_index()
    result.columns = ['stock_name', 'capital_gain_loss']
    return result

# Alternative approach
def capital_gainloss_v2(stocks: pd.DataFrame) -> pd.DataFrame:
    buy = stocks[stocks['operation'] == 'Buy'].groupby('stock_name')['price'].sum()
    sell = stocks[stocks['operation'] == 'Sell'].groupby('stock_name')['price'].sum()

    result = (sell - buy).reset_index()
    result.columns = ['stock_name', 'capital_gain_loss']
    return result


# ============================================================================
# 1454. Active Users
# ============================================================================

def active_users(accounts: pd.DataFrame, logins: pd.DataFrame) -> pd.DataFrame:
    """Find users active for 5+ consecutive days"""
    # Get unique login dates per user
    logins = logins[['id', 'login_date']].drop_duplicates().sort_values(['id', 'login_date'])

    # Create group for consecutive dates
    logins['date_int'] = (logins['login_date'] - pd.Timestamp('1970-01-01')).dt.days
    logins['group'] = logins['date_int'] - logins.groupby('id').cumcount()

    # Count consecutive days
    consecutive = logins.groupby(['id', 'group']).size().reset_index(name='consecutive_days')
    active_user_ids = consecutive[consecutive['consecutive_days'] >= 5]['id'].unique()

    # Get account info
    result = accounts[accounts['id'].isin(active_user_ids)][['id', 'name']]
    return result.sort_values('id')


# ============================================================================
# 603. Consecutive Available Seats
# ============================================================================

def consecutive_seats(cinema: pd.DataFrame) -> pd.DataFrame:
    """Find consecutive available seats"""
    cinema = cinema.sort_values('seat_id')
    cinema['prev_free'] = cinema['free'].shift(1)
    cinema['next_free'] = cinema['free'].shift(-1)

    result = cinema[
        (cinema['free'] == 1) &
        ((cinema['prev_free'] == 1) | (cinema['next_free'] == 1))
    ][['seat_id']]

    return result.sort_values('seat_id')


# ============================================================================
# 1336. Number of Transactions per Visit
# ============================================================================

def transactions_per_visit(visits: pd.DataFrame, transactions: pd.DataFrame) -> pd.DataFrame:
    """Count visits by number of transactions"""
    # Merge visits with transactions
    merged = visits.merge(
        transactions,
        left_on=['user_id', 'visit_date'],
        right_on=['user_id', 'transaction_date'],
        how='left'
    )

    # Count transactions per visit
    visit_counts = merged.groupby(['user_id', 'visit_date']).size().reset_index(name='transactions_count')
    visit_counts['transactions_count'] = visit_counts['transactions_count'].fillna(0).astype(int)

    # Count visits by transaction count
    result = visit_counts.groupby('transactions_count').size().reset_index(name='visits_count')

    # Fill in missing transaction counts
    max_trans = result['transactions_count'].max() if not result.empty else 0
    all_counts = pd.DataFrame({'transactions_count': range(max_trans + 1)})
    result = all_counts.merge(result, on='transactions_count', how='left')
    result['visits_count'] = result['visits_count'].fillna(0).astype(int)

    return result


# ============================================================================
# 1369. Get the Second Most Recent Activity
# ============================================================================

def second_most_recent_activity(user_activity: pd.DataFrame) -> pd.DataFrame:
    """Get second most recent activity (or only one if user has just one)"""
    # Rank activities per user
    user_activity['rank'] = user_activity.groupby('username')['startDate'].rank(
        method='dense', ascending=False
    )

    # Count activities per user
    activity_counts = user_activity.groupby('username').size().reset_index(name='count')
    user_activity = user_activity.merge(activity_counts, on='username')

    # Get second most recent, or first if only one activity
    result = user_activity[
        (user_activity['rank'] == 2) |
        ((user_activity['count'] == 1) & (user_activity['rank'] == 1))
    ]

    return result[['username', 'activity', 'startDate', 'endDate']]


# ============================================================================
# 1412. Find the Quiet Students in All Exams
# ============================================================================

def quiet_students(student: pd.DataFrame, exam: pd.DataFrame) -> pd.DataFrame:
    """Find students who are never top or bottom scorer in any exam"""
    # Find max and min scores per exam
    exam_stats = exam.groupby('exam_id')['score'].agg(['max', 'min']).reset_index()

    # Find students who scored max or min in any exam
    noisy = exam.merge(exam_stats, on='exam_id')
    noisy = noisy[(noisy['score'] == noisy['max']) | (noisy['score'] == noisy['min'])]
    noisy_students = noisy['student_id'].unique()

    # Find students who took exams but are not noisy
    exam_takers = exam['student_id'].unique()
    quiet_ids = [sid for sid in exam_takers if sid not in noisy_students]

    result = student[student['student_id'].isin(quiet_ids)][['student_id', 'student_name']]
    return result.sort_values('student_id')


# ============================================================================
# Example Usage
# ============================================================================

if __name__ == "__main__":
    # Test Nth Highest Salary
    employee_df = pd.DataFrame({
        'id': [1, 2, 3],
        'salary': [100, 200, 300]
    })
    print("Nth Highest Salary (N=2):")
    print(nth_highest_salary(employee_df, 2))
    print()

    # Test Rank Scores
    scores_df = pd.DataFrame({
        'id': [1, 2, 3, 4, 5],
        'score': [3.50, 3.65, 4.00, 3.85, 4.00]
    })
    print("Rank Scores:")
    print(rank_scores(scores_df))
    print()

    # Test Consecutive Numbers
    logs_df = pd.DataFrame({
        'id': [1, 2, 3, 4, 5, 6, 7],
        'num': [1, 1, 1, 2, 1, 2, 2]
    })
    print("Consecutive Numbers:")
    print(consecutive_numbers(logs_df))

In [ ]:
-- ============================================================================
-- EASY PROBLEMS
-- ============================================================================

-- 1757. Recyclable and Low Fat Products
SELECT product_id
FROM Products
WHERE low_fats = 'Y' AND recyclable = 'Y';

-- 584. Find Customer Referee
SELECT name
FROM Customer
WHERE referee_id IS NULL OR referee_id != 2;

-- 595. Big Countries
SELECT name, population, area
FROM World
WHERE area >= 3000000 OR population >= 25000000;

-- 1148. Article Views I
SELECT DISTINCT author_id as id
FROM Views
WHERE author_id = viewer_id
ORDER BY id;

-- 1683. Invalid Tweets
SELECT tweet_id
FROM Tweets
WHERE LENGTH(content) > 15;

-- 1378. Replace Employee ID With The Unique Identifier
SELECT u.unique_id, e.name
FROM Employees e
LEFT JOIN EmployeeUNI u ON e.id = u.id;

-- 1068. Product Sales Analysis I
SELECT p.product_name, s.year, s.price
FROM Sales s
JOIN Product p ON s.product_id = p.product_id;

-- 1581. Customer Who Visited but Did Not Make Any Transactions
SELECT v.customer_id, COUNT(*) as count_no_trans
FROM Visits v
LEFT JOIN Transactions t ON v.visit_id = t.visit_id
WHERE t.transaction_id IS NULL
GROUP BY v.customer_id;

-- 197. Rising Temperature
SELECT w1.id
FROM Weather w1
JOIN Weather w2
    ON w1.recordDate = w2.recordDate + INTERVAL '1' DAY
WHERE w1.temperature > w2.temperature;

-- 1661. Average Time of Process per Machine
SELECT machine_id,
       ROUND(AVG(end_time - start_time), 3) as processing_time
FROM (
    SELECT machine_id, process_id,
           MAX(CASE WHEN activity_type = 'end' THEN timestamp END) as end_time,
           MAX(CASE WHEN activity_type = 'start' THEN timestamp END) as start_time
    FROM Activity
    GROUP BY machine_id, process_id
)
GROUP BY machine_id;

-- 577. Employee Bonus
SELECT e.name, b.bonus
FROM Employee e
LEFT JOIN Bonus b ON e.empId = b.empId
WHERE b.bonus < 1000 OR b.bonus IS NULL;

-- 1280. Students and Examinations
SELECT s.student_id, s.student_name, sub.subject_name,
       COUNT(e.subject_name) as attended_exams
FROM Students s
CROSS JOIN Subjects sub
LEFT JOIN Examinations e
    ON s.student_id = e.student_id
    AND sub.subject_name = e.subject_name
GROUP BY s.student_id, s.student_name, sub.subject_name
ORDER BY s.student_id, sub.subject_name;

-- 570. Managers with at Least 5 Direct Reports
SELECT name
FROM Employee
WHERE id IN (
    SELECT managerId
    FROM Employee
    WHERE managerId IS NOT NULL
    GROUP BY managerId
    HAVING COUNT(*) >= 5
);

-- 1934. Confirmation Rate
SELECT s.user_id,
       ROUND(
           COALESCE(
               SUM(CASE WHEN c.action = 'confirmed' THEN 1 ELSE 0 END) /
               NULLIF(COUNT(c.action), 0),
               0
           ), 2
       ) as confirmation_rate
FROM Signups s
LEFT JOIN Confirmations c ON s.user_id = c.user_id
GROUP BY s.user_id;

-- 620. Not Boring Movies
SELECT *
FROM Cinema
WHERE MOD(id, 2) = 1 AND description != 'boring'
ORDER BY rating DESC;

-- 1251. Average Selling Price
SELECT p.product_id,
       COALESCE(
           ROUND(SUM(u.units * p.price) / NULLIF(SUM(u.units), 0), 2),
           0
       ) as average_price
FROM Prices p
LEFT JOIN UnitsSold u
    ON p.product_id = u.product_id
    AND u.purchase_date BETWEEN p.start_date AND p.end_date
GROUP BY p.product_id;

-- 1075. Project Employees I
SELECT p.project_id,
       ROUND(AVG(e.experience_years), 2) as average_years
FROM Project p
JOIN Employee e ON p.employee_id = e.employee_id
GROUP BY p.project_id;

-- 1633. Percentage of Users Attended a Contest
SELECT contest_id,
       ROUND(COUNT(DISTINCT user_id) * 100.0 / (SELECT COUNT(*) FROM Users), 2) as percentage
FROM Register
GROUP BY contest_id
ORDER BY percentage DESC, contest_id;

-- 1211. Queries Quality and Percentage
SELECT query_name,
       ROUND(AVG(rating * 1.0 / position), 2) as quality,
       ROUND(SUM(CASE WHEN rating < 3 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as poor_query_percentage
FROM Queries
WHERE query_name IS NOT NULL
GROUP BY query_name;

-- 1193. Monthly Transactions I
SELECT TO_CHAR(trans_date, 'YYYY-MM') as month,
       country,
       COUNT(*) as trans_count,
       SUM(CASE WHEN state = 'approved' THEN 1 ELSE 0 END) as approved_count,
       SUM(amount) as trans_total_amount,
       SUM(CASE WHEN state = 'approved' THEN amount ELSE 0 END) as approved_total_amount
FROM Transactions
GROUP BY TO_CHAR(trans_date, 'YYYY-MM'), country;

-- 1174. Immediate Food Delivery II
SELECT ROUND(
    SUM(CASE WHEN order_date = customer_pref_delivery_date THEN 1 ELSE 0 END) * 100.0 /
    COUNT(*), 2
) as immediate_percentage
FROM Delivery
WHERE (customer_id, order_date) IN (
    SELECT customer_id, MIN(order_date)
    FROM Delivery
    GROUP BY customer_id
);

-- 550. Game Play Analysis IV
SELECT ROUND(
    COUNT(DISTINCT player_id) * 1.0 /
    (SELECT COUNT(DISTINCT player_id) FROM Activity), 2
) as fraction
FROM Activity
WHERE (player_id, event_date) IN (
    SELECT player_id, MIN(event_date) + INTERVAL '1' DAY
    FROM Activity
    GROUP BY player_id
)
AND (player_id, event_date - INTERVAL '1' DAY) IN (
    SELECT player_id, event_date FROM Activity
);

-- ============================================================================
-- MEDIUM PROBLEMS - Grouping and Aggregation
-- ============================================================================

-- 1141. User Activity for the Past 30 Days I
SELECT activity_date as day,
       COUNT(DISTINCT user_id) as active_users
FROM Activity
WHERE activity_date BETWEEN DATE '2019-07-27' - INTERVAL '29' DAY AND DATE '2019-07-27'
GROUP BY activity_date;

-- 1070. Product Sales Analysis III
SELECT product_id, year as first_year, quantity, price
FROM Sales
WHERE (product_id, year) IN (
    SELECT product_id, MIN(year)
    FROM Sales
    GROUP BY product_id
);

-- 596. Classes With at Least 5 Students
SELECT class
FROM Courses
GROUP BY class
HAVING COUNT(DISTINCT student) >= 5;

-- 1084. Sales Analysis III
SELECT p.product_id, p.product_name
FROM Product p
WHERE p.product_id IN (
    SELECT product_id
    FROM Sales
    GROUP BY product_id
    HAVING MIN(sale_date) >= DATE '2019-01-01'
       AND MAX(sale_date) <= DATE '2019-03-31'
)
AND p.product_id NOT IN (
    SELECT product_id
    FROM Sales
    WHERE sale_date < DATE '2019-01-01'
       OR sale_date > DATE '2019-03-31'
);

-- 1729. Find Followers Count
SELECT user_id, COUNT(follower_id) as followers_count
FROM Followers
GROUP BY user_id
ORDER BY user_id;

-- 619. Biggest Single Number
SELECT MAX(num) as num
FROM (
    SELECT num
    FROM MyNumbers
    GROUP BY num
    HAVING COUNT(*) = 1
);

-- 1045. Customers Who Bought All Products
SELECT customer_id
FROM Customer
GROUP BY customer_id
HAVING COUNT(DISTINCT product_key) = (SELECT COUNT(*) FROM Product);

-- ============================================================================
-- MEDIUM PROBLEMS - Advanced Joins
-- ============================================================================

-- 1731. The Number of Employees Which Report to Each Employee
SELECT e1.employee_id,
       e1.name,
       COUNT(e2.employee_id) as reports_count,
       ROUND(AVG(e2.age)) as average_age
FROM Employees e1
JOIN Employees e2 ON e1.employee_id = e2.reports_to
GROUP BY e1.employee_id, e1.name
ORDER BY e1.employee_id;

-- 1789. Primary Department for Each Employee
SELECT employee_id, department_id
FROM Employee
WHERE primary_flag = 'Y'
   OR employee_id IN (
       SELECT employee_id
       FROM Employee
       GROUP BY employee_id
       HAVING COUNT(*) = 1
   );

-- 610. Triangle Judgement
SELECT x, y, z,
       CASE
           WHEN x + y > z AND x + z > y AND y + z > x THEN 'Yes'
           ELSE 'No'
       END as triangle
FROM Triangle;

-- 1204. Last Person to Fit in the Bus
SELECT person_name
FROM (
    SELECT person_name,
           SUM(weight) OVER (ORDER BY turn) as cumulative_weight
    FROM Queue
)
WHERE cumulative_weight <= 1000
ORDER BY cumulative_weight DESC
FETCH FIRST 1 ROW ONLY;

-- 1907. Count Salary Categories
SELECT 'Low Salary' as category,
       SUM(CASE WHEN income < 20000 THEN 1 ELSE 0 END) as accounts_count
FROM Accounts
UNION ALL
SELECT 'Average Salary' as category,
       SUM(CASE WHEN income BETWEEN 20000 AND 50000 THEN 1 ELSE 0 END) as accounts_count
FROM Accounts
UNION ALL
SELECT 'High Salary' as category,
       SUM(CASE WHEN income > 50000 THEN 1 ELSE 0 END) as accounts_count
FROM Accounts;

-- 1978. Employees Whose Manager Left the Company
SELECT employee_id
FROM Employees
WHERE salary < 30000
  AND manager_id IS NOT NULL
  AND manager_id NOT IN (SELECT employee_id FROM Employees)
ORDER BY employee_id;

-- ============================================================================
-- STRING FUNCTIONS
-- ============================================================================

-- 1667. Fix Names in a Table
SELECT user_id,
       INITCAP(name) as name
FROM Users
ORDER BY user_id;

-- 1527. Patients With a Condition
SELECT *
FROM Patients
WHERE conditions LIKE 'DIAB1%'
   OR conditions LIKE '% DIAB1%';

-- 176. Second Highest Salary
SELECT MAX(salary) as SecondHighestSalary
FROM Employee
WHERE salary < (SELECT MAX(salary) FROM Employee);

-- 1484. Group Sold Products By The Date
SELECT sell_date,
       COUNT(DISTINCT product) as num_sold,
       LISTAGG(DISTINCT product, ',') WITHIN GROUP (ORDER BY product) as products
FROM Activities
GROUP BY sell_date
ORDER BY sell_date;

-- 1327. List the Products Ordered in a Period
SELECT p.product_name,
       SUM(o.unit) as unit
FROM Products p
JOIN Orders o ON p.product_id = o.product_id
WHERE TO_CHAR(o.order_date, 'YYYY-MM') = '2020-02'
GROUP BY p.product_name
HAVING SUM(o.unit) >= 100;

-- 1517. Find Users With Valid E-Mails
SELECT *
FROM Users
WHERE REGEXP_LIKE(mail, '^[A-Za-z][A-Za-z0-9_.-]*@leetcode\.com$');

In [ ]:
-- ============================================================================
-- ADDITIONAL MEDIUM/HARD PROBLEMS
-- ============================================================================

-- 1158. Market Analysis I
SELECT u.user_id as buyer_id,
       u.join_date,
       COALESCE(COUNT(o.order_id), 0) as orders_in_2019
FROM Users u
LEFT JOIN Orders o
    ON u.user_id = o.buyer_id
    AND EXTRACT(

In [ ]:
-- ============================================================================
-- ADDITIONAL MEDIUM/HARD PROBLEMS
-- ============================================================================

-- 1158. Market Analysis I
SELECT u.user_id as buyer_id,
       u.join_date,
       COALESCE(COUNT(o.order_id), 0) as orders_in_2019
FROM Users u
LEFT JOIN Orders o
    ON u.user_id = o.buyer_id
    AND EXTRACT(YEAR FROM o.order_date) = 2019
GROUP BY u.user_id, u.join_date;

-- 607. Sales Person
SELECT name
FROM SalesPerson
WHERE sales_id NOT IN (
    SELECT DISTINCT s.sales_id
    FROM Orders o
    JOIN Company c ON o.com_id = c.com_id
    JOIN SalesPerson s ON o.sales_id = s.sales_id
    WHERE c.name = 'RED'
);

-- 608. Tree Node
SELECT id,
       CASE
           WHEN p_id IS NULL THEN 'Root'
           WHEN id IN (SELECT DISTINCT p_id FROM Tree WHERE p_id IS NOT NULL) THEN 'Inner'
           ELSE 'Leaf'
       END as type
FROM Tree;

-- 1741. Find Total Time Spent by Each Employee
SELECT event_day as day,
       emp_id,
       SUM(out_time - in_time) as total_time
FROM Employees
GROUP BY event_day, emp_id;

-- 1873. Calculate Special Bonus
SELECT employee_id,
       CASE
           WHEN MOD(employee_id, 2) = 1 AND name NOT LIKE 'M%' THEN salary
           ELSE 0
       END as bonus
FROM Employees
ORDER BY employee_id;

-- 1050. Actors and Directors Who Cooperated At Least Three Times
SELECT actor_id, director_id
FROM ActorDirector
GROUP BY actor_id, director_id
HAVING COUNT(*) >= 3;

-- 627. Swap Salary
UPDATE Salary
SET sex = CASE
    WHEN sex = 'm' THEN 'f'
    WHEN sex = 'f' THEN 'm'
END;

-- Alternative without UPDATE (for SELECT query)
SELECT emp_name,
       CASE
           WHEN sex = 'm' THEN 'f'
           ELSE 'm'
       END as sex,
       salary
FROM Salary;

-- 1667. Fix Names in a Table (Oracle version)
SELECT user_id,
       UPPER(SUBSTR(name, 1, 1)) || LOWER(SUBSTR(name, 2)) as name
FROM Users
ORDER BY user_id;

-- 1068. Product Sales Analysis I
SELECT p.product_name, s.year, s.price
FROM Sales s
JOIN Product p ON s.product_id = p.product_id;

-- 1965. Employees With Missing Information
SELECT employee_id
FROM (
    SELECT employee_id FROM Employees
    UNION
    SELECT employee_id FROM Salaries
)
WHERE employee_id NOT IN (
    SELECT e.employee_id
    FROM Employees e
    JOIN Salaries s ON e.employee_id = s.employee_id
)
ORDER BY employee_id;

-- 181. Employees Earning More Than Their Managers
SELECT e1.name as Employee
FROM Employee e1
JOIN Employee e2 ON e1.managerId = e2.id
WHERE e1.salary > e2.salary;

-- 182. Duplicate Emails
SELECT email
FROM Person
GROUP BY email
HAVING COUNT(*) > 1;

-- 183. Customers Who Never Order
SELECT name as Customers
FROM Customers
WHERE id NOT IN (SELECT DISTINCT customerId FROM Orders);

-- 1965. Bank Account Summary II
SELECT u.name, SUM(t.amount) as balance
FROM Users u
JOIN Transactions t ON u.account = t.account
GROUP BY u.name
HAVING SUM(t.amount) > 10000;

-- 512. Game Play Analysis I
SELECT player_id, MIN(event_date) as first_login
FROM Activity
GROUP BY player_id;

-- 1350. Students With Invalid Departments
SELECT s.id, s.name
FROM Students s
LEFT JOIN Departments d ON s.department_id = d.id
WHERE d.id IS NULL;

-- 1587. Bank Account Summary II (Full version)
SELECT u.name, SUM(t.amount) as balance
FROM Users u
JOIN Transactions t ON u.account = t.account
GROUP BY u.account, u.name
HAVING SUM(t.amount) > 10000;

-- 1407. Top Travellers
SELECT u.name,
       COALESCE(SUM(r.distance), 0) as travelled_distance
FROM Users u
LEFT JOIN Rides r ON u.id = r.user_id
GROUP BY u.id, u.name
ORDER BY travelled_distance DESC, u.name;

-- 1965. Employees With Missing Information (Better version)
SELECT t.employee_id
FROM (
    SELECT employee_id FROM Employees
    UNION ALL
    SELECT employee_id FROM Salaries
) t
GROUP BY t.employee_id
HAVING COUNT(t.employee_id) = 1
ORDER BY t.employee_id;

-- 262. Trips and Users
SELECT request_at as Day,
       ROUND(
           SUM(CASE WHEN status LIKE 'cancelled%' THEN 1 ELSE 0 END) * 1.0 / COUNT(*),
           2
       ) as "Cancellation Rate"
FROM Trips t
JOIN Users client ON t.client_id = client.users_id
JOIN Users driver ON t.driver_id = driver.users_id
WHERE client.banned = 'No'
  AND driver.banned = 'No'
  AND request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY request_at;

-- 1179. Reformat Department Table
SELECT id,
       SUM(CASE WHEN month = 'Jan' THEN revenue END) as Jan_Revenue,
       SUM(CASE WHEN month = 'Feb' THEN revenue END) as Feb_Revenue,
       SUM(CASE WHEN month = 'Mar' THEN revenue END) as Mar_Revenue,
       SUM(CASE WHEN month = 'Apr' THEN revenue END) as Apr_Revenue,
       SUM(CASE WHEN month = 'May' THEN revenue END) as May_Revenue,
       SUM(CASE WHEN month = 'Jun' THEN revenue END) as Jun_Revenue,
       SUM(CASE WHEN month = 'Jul' THEN revenue END) as Jul_Revenue,
       SUM(CASE WHEN month = 'Aug' THEN revenue END) as Aug_Revenue,
       SUM(CASE WHEN month = 'Sep' THEN revenue END) as Sep_Revenue,
       SUM(CASE WHEN month = 'Oct' THEN revenue END) as Oct_Revenue,
       SUM(CASE WHEN month = 'Nov' THEN revenue END) as Nov_Revenue,
       SUM(CASE WHEN month = 'Dec' THEN revenue END) as Dec_Revenue
FROM Department
GROUP BY id;

-- 1327. List the Products Ordered in a Period
SELECT p.product_name, SUM(o.unit) as unit
FROM Products p
JOIN Orders o ON p.product_id = o.product_id
WHERE o.order_date BETWEEN DATE '2020-02-01' AND DATE '2020-02-29'
GROUP BY p.product_id, p.product_name
HAVING SUM(o.unit) >= 100;

-- 1179. Daily Leads and Partners
SELECT date_id, make_name,
       COUNT(DISTINCT lead_id) as unique_leads,
       COUNT(DISTINCT partner_id) as unique_partners
FROM DailySales
GROUP BY date_id, make_name;

-- 175. Combine Two Tables
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person p
LEFT JOIN Address a ON p.personId = a.personId;

-- 1890. The Latest Login in 2020
SELECT user_id, MAX(time_stamp) as last_stamp
FROM Logins
WHERE EXTRACT(YEAR FROM time_stamp) = 2020
GROUP BY user_id;

-- 1050. Rearrange Products Table
SELECT product_id, 'store1' as store, store1 as price
FROM Products
WHERE store1 IS NOT NULL
UNION ALL
SELECT product_id, 'store2' as store, store2 as price
FROM Products
WHERE store2 IS NOT NULL
UNION ALL
SELECT product_id, 'store3' as store, store3 as price
FROM Products
WHERE store3 IS NOT NULL;

-- 1795. Rearrange Products Table (Alternative)
SELECT product_id, store, price
FROM Products
UNPIVOT (
    price FOR store IN (store1, store2, store3)
);

-- ============================================================================
-- ADVANCED ANALYTICS PROBLEMS
-- ============================================================================

-- 1205. Monthly Transactions II
WITH all_trans AS (
    SELECT id, country, state, amount, trans_date, 'trans' as type
    FROM Transactions
    UNION ALL
    SELECT trans_id as id, country, 'chargeback' as state, amount, trans_date, 'chargeback' as type
    FROM Chargebacks c
    JOIN Transactions t ON c.trans_id = t.id
)
SELECT TO_CHAR(trans_date, 'YYYY-MM') as month,
       country,
       SUM(CASE WHEN type = 'trans' AND state = 'approved' THEN 1 ELSE 0 END) as approved_count,
       SUM(CASE WHEN type = 'trans' AND state = 'approved' THEN amount ELSE 0 END) as approved_amount,
       SUM(CASE WHEN type = 'chargeback' THEN 1 ELSE 0 END) as chargeback_count,
       SUM(CASE WHEN type = 'chargeback' THEN amount ELSE 0 END) as chargeback_amount
FROM all_trans
GROUP BY TO_CHAR(trans_date, 'YYYY-MM'), country
HAVING SUM(CASE WHEN type = 'trans' AND state = 'approved' THEN 1 ELSE 0 END) > 0
    OR SUM(CASE WHEN type = 'chargeback' THEN 1 ELSE 0 END) > 0;

-- 602. Friend Requests II: Who Has the Most Friends
SELECT id, COUNT(*) as num
FROM (
    SELECT requester_id as id FROM RequestAccepted
    UNION ALL
    SELECT accepter_id as id FROM RequestAccepted
)
GROUP BY id
ORDER BY num DESC
FETCH FIRST 1 ROW ONLY;

-- 585. Investments in 2016
SELECT ROUND(SUM(tiv_2016), 2) as tiv_2016
FROM Insurance
WHERE tiv_2015 IN (
    SELECT tiv_2015
    FROM Insurance
    GROUP BY tiv_2015
    HAVING COUNT(*) > 1
)
AND (lat, lon) IN (
    SELECT lat, lon
    FROM Insurance
    GROUP BY lat, lon
    HAVING COUNT(*) = 1
);

-- 1164. Product Price at a Given Date
SELECT product_id,
       COALESCE(
           (SELECT new_price
            FROM Products p2
            WHERE p2.product_id = p1.product_id
              AND p2.change_date <= DATE '2019-08-16'
            ORDER BY p2.change_date DESC
            FETCH FIRST 1 ROW ONLY),
           10
       ) as price
FROM (SELECT DISTINCT product_id FROM Products) p1;

-- 1193. Restaurant Growth
SELECT visited_on,
       amount,
       ROUND(amount / 7, 2) as average_amount
FROM (
    SELECT DISTINCT visited_on,
           SUM(amount) OVER (
               ORDER BY visited_on
               ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
           ) as amount,
           ROW_NUMBER() OVER (ORDER BY visited_on) as rn
    FROM (
        SELECT visited_on, SUM(amount) as amount
        FROM Customer
        GROUP BY visited_on
    )
)
WHERE rn >= 7;

-- 1321. Restaurant Growth (Alternative)
SELECT c1.visited_on,
       SUM(c2.amount) as amount,
       ROUND(AVG(c2.amount), 2) as average_amount
FROM (SELECT DISTINCT visited_on FROM Customer) c1
JOIN (
    SELECT visited_on, SUM(amount) as amount
    FROM Customer
    GROUP BY visited_on
) c2
ON c2.visited_on BETWEEN c1.visited_on - INTERVAL '6' DAY AND c1.visited_on
WHERE c1.visited_on >= (SELECT MIN(visited_on) + INTERVAL '6' DAY FROM Customer)
GROUP BY c1.visited_on
ORDER BY c1.visited_on;

-- 1264. Page Recommendations
SELECT DISTINCT page_id as recommended_page
FROM Likes
WHERE user_id IN (
    SELECT user2_id FROM Friendship WHERE user1_id = 1
    UNION
    SELECT user1_id FROM Friendship WHERE user2_id = 1
)
AND page_id NOT IN (
    SELECT page_id FROM Likes WHERE user_id = 1
);

-- ============================================================================
-- WINDOW FUNCTIONS AND ADVANCED QUERIES
-- ============================================================================

-- 1308. Running Total for Different Genders
SELECT gender, day,
       SUM(score_points) OVER (PARTITION BY gender ORDER BY day) as total
FROM Scores
ORDER BY gender, day;

-- 1270. All People Report to the Given Manager
SELECT e1.employee_id
FROM Employees e1
JOIN Employees e2 ON e1.manager_id = e2.employee_id
JOIN Employees e3 ON e2.manager_id = e3.employee_id
WHERE e3.manager_id = 1
  AND e1.employee_id != 1;

-- 1097. Game Play Analysis V
WITH first_login AS (
    SELECT player_id, MIN(event_date) as first_date
    FROM Activity
    GROUP BY player_id
)
SELECT f.first_date as install_dt,
       COUNT(DISTINCT f.player_id) as installs,
       ROUND(
           COUNT(DISTINCT CASE
               WHEN a.event_date = f.first_date + INTERVAL '1' DAY
               THEN a.player_id
           END) * 1.0 / COUNT(DISTINCT f.player_id),
           2
       ) as Day1_retention
FROM first_login f
LEFT JOIN Activity a
    ON f.player_id = a.player_id
    AND a.event_date BETWEEN f.first_date AND f.first_date + INTERVAL '1' DAY
GROUP BY f.first_date;

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ============================================================================
# EASY PROBLEMS - Python Solutions
# ============================================================================

def recyclable_and_low_fat(products: pd.DataFrame) -> pd.DataFrame:
    """Select recyclable and low fat products"""
    return products[
        (products['low_fats'] == 'Y') &
        (products['recyclable'] == 'Y')
    ][['product_id']]


def find_customer_referee(customer: pd.DataFrame) -> pd.DataFrame:
    """Find customers not referred by customer 2"""
    return customer[
        customer['referee_id'].isna() |
        (customer['referee_id'] != 2)
    ][['name']]


def big_countries(world: pd.DataFrame) -> pd.DataFrame:
    """Find big countries (area >= 3M or population >= 25M)"""
    return world[
        (world['area'] >= 3000000) |
        (world['population'] >= 25000000)
    ][['name', 'population', 'area']]


def article_views(views: pd.DataFrame) -> pd.DataFrame:
    """Find authors who viewed their own articles"""
    result = views[views['author_id'] == views['viewer_id']][['author_id']]
    result = result.drop_duplicates().rename(columns={'author_id': 'id'})
    return result.sort_values('id')


def invalid_tweets(tweets: pd.DataFrame) -> pd.DataFrame:
    """Find tweets with content length > 15"""
    return tweets[tweets['content'].str.len() > 15][['tweet_id']]


def replace_employee_id(employees: pd.DataFrame, employee_uni: pd.DataFrame) -> pd.DataFrame:
    """Replace employee ID with unique identifier"""
    return employees.merge(
        employee_uni,
        left_on='id',
        right_on='id',
        how='left'
    )[['unique_id', 'name']]


def sales_analysis(sales: pd.DataFrame, product: pd.DataFrame) -> pd.DataFrame:
    """Product sales analysis"""
    return sales.merge(product, on='product_id')[['product_name', 'year', 'price']]


def no_transactions(visits: pd.DataFrame, transactions: pd.DataFrame) -> pd.DataFrame:
    """Customers who visited but made no transactions"""
    merged = visits.merge(transactions, on='visit_id', how='left')
    no_trans = merged[merged['transaction_id'].isna()]
    result = no_trans.groupby('customer_id').size().reset_index(name='count_no_trans')
    return result


def rising_temperature(weather: pd.DataFrame) -> pd.DataFrame:
    """Find days with temperature higher than previous day"""
    weather = weather.sort_values('recordDate')
    weather['prev_temp'] = weather['temperature'].shift(1)
    weather['prev_date'] = weather['recordDate'].shift(1)
    weather['date_diff'] = (weather['recordDate'] - weather['prev_date']).dt.days

    result = weather[
        (weather['temperature'] > weather['prev_temp']) &
        (weather['date_diff'] == 1)
    ][['id']]

    return result


def average_time_process(activity: pd.DataFrame) -> pd.DataFrame:
    """Calculate average processing time per machine"""
    # Pivot to get start and end times
    start = activity[activity['activity_type'] == 'start'][['machine_id', 'process_id', 'timestamp']]
    end = activity[activity['activity_type'] == 'end'][['machine_id', 'process_id', 'timestamp']]

    merged = start.merge(
        end,
        on=['machine_id', 'process_id'],
        suffixes=('_start', '_end')
    )
    merged['processing_time'] = merged['timestamp_end'] - merged['timestamp_start']

    result = merged.groupby('machine_id')['processing_time'].mean().reset_index()
    result['processing_time'] = result['processing_time'].round(3)

    return result


def employee_bonus(employee: pd.DataFrame, bonus: pd.DataFrame) -> pd.DataFrame:
    """Find employees with bonus < 1000 or no bonus"""
    merged = employee.merge(bonus, on='empId', how='left')
    result = merged[(merged['bonus'] < 1000) | merged['bonus'].isna()]
    return result[['name', 'bonus']]


def students_and_examinations(students: pd.DataFrame, subjects: pd.DataFrame,
                              examinations: pd.DataFrame) -> pd.DataFrame:
    """Count exam attendance per student and subject"""
    # Cross join students and subjects
    cross = students.merge(subjects, how='cross')

    # Count examinations
    exam_counts = examinations.groupby(['student_id', 'subject_name']).size().reset_index(name='attended_exams')

    # Merge with cross product
    result = cross.merge(
        exam_counts,
        on=['student_id', 'subject_name'],
        how='left'
    )
    result['attended_exams'] = result['attended_exams'].fillna(0).astype(int)

    return result[['student_id', 'student_name', 'subject_name', 'attended_exams']].sort_values(
        ['student_id', 'subject_name']
    )


def managers_with_five_reports(employee: pd.DataFrame) -> pd.DataFrame:
    """Find managers with at least 5 direct reports"""
    reports = employee['managerId'].value_counts()
    manager_ids = reports[reports >= 5].index

    return employee[employee['id'].isin(manager_ids)][['name']]


def confirmation_rate(signups: pd.DataFrame, confirmations: pd.DataFrame) -> pd.DataFrame:
    """Calculate confirmation rate per user"""
    # Count confirmations per user
    conf_stats = confirmations.groupby('user_id').agg(
        total_actions=('action', 'count'),
        confirmed_actions=('action', lambda x: (x == 'confirmed').sum())
    ).reset_index()

    # Merge with signups
    result = signups.merge(conf_stats, on='user_id', how='left')
    result['total_actions'] = result['total_actions'].fillna(0)
    result['confirmed_actions'] = result['confirmed_actions'].fillna(0)

    # Calculate rate
    result['confirmation_rate'] = (
        result['confirmed_actions'] / result['total_actions'].replace(0, 1)
    ).fillna(0).round(2)

    return result[['user_id', 'confirmation_rate']]


def not_boring_movies(cinema: pd.DataFrame) -> pd.DataFrame:
    """Find not boring movies with odd ID"""
    result = cinema[
        (cinema['id'] % 2 == 1) &
        (cinema['description'] != 'boring')
    ]
    return result.sort_values('rating', ascending=False)


def average_selling_price(prices: pd.DataFrame, units_sold: pd.DataFrame) -> pd.DataFrame:
    """Calculate average selling price"""
    # Merge on product and date range
    merged = prices.merge(units_sold, on='product_id', how='left')
    merged = merged[
        (merged['purchase_date'].isna()) |
        ((merged['purchase_date'] >= merged['start_date']) &
         (merged['purchase_date'] <= merged['end_date']))
    ]

    # Calculate weighted average
    merged['total_value'] = merged['units'].fillna(0) * merged['price']

    result = merged.groupby('product_id').agg(
        total_value=('total_value', 'sum'),
        total_units=('units', 'sum')
    ).reset_index()

    result['average_price'] = (
        result['total_value'] / result['total_units'].replace(0, 1)
    ).fillna(0).round(2)

    return result[['product_id', 'average_price']]


def project_employees(project: pd.DataFrame, employee: pd.DataFrame) -> pd.DataFrame:
    """Calculate average years of experience per project"""
    merged = project.merge(employee, on='employee_id')
    result = merged.groupby('project_id')['experience_years'].mean().reset_index()
    result['experience_years'] = result['experience_years'].round(2)
    result.columns = ['project_id', 'average_years']
    return result


def contest_attendance(users: pd.DataFrame, register: pd.DataFrame) -> pd.DataFrame:
    """Calculate percentage of users attending each contest"""
    total_users = len(users)

    result = register.groupby('contest_id')['user_id'].nunique().reset_index()
    result['percentage'] = (result['user_id'] / total_users * 100).round(2)
    result = result.rename(columns={'user_id': 'registered_users'})

    return result[['contest_id', 'percentage']].sort_values(
        ['percentage', 'contest_id'],
        ascending=[False, True]
    )


def queries_quality(queries: pd.DataFrame) -> pd.DataFrame:
    """Calculate query quality and poor query percentage"""
    queries = queries[queries['query_name'].notna()]

    # Calculate quality
    queries['quality_score'] = queries['rating'] / queries['position']
    queries['is_poor'] = (queries['rating'] < 3).astype(int)

    result = queries.groupby('query_name').agg(
        quality=('quality_score', 'mean'),
        poor_query_percentage=('is_poor', lambda x: x.mean() * 100)
    ).reset_index()

    result['quality'] = result['quality'].round(2)
    result['poor_query_percentage'] = result['poor_query_percentage'].round(2)

    return result


def monthly_transactions(transactions: pd.DataFrame) -> pd.DataFrame:
    """Calculate monthly transaction statistics"""
    transactions['month'] = transactions['trans_date'].dt.to_period('M').astype(str)

    result = transactions.groupby(['month', 'country']).agg(
        trans_count=('id', 'count'),
        approved_count=('state', lambda x: (x == 'approved').sum()),
        trans_total_amount=('amount', 'sum'),
        approved_total_amount=('amount', lambda x: x[transactions.loc[x.index, 'state'] == 'approved'].sum())
    ).reset_index()

    return result


def immediate_food_delivery(delivery: pd.DataFrame) -> pd.DataFrame:
    """Calculate percentage of immediate first orders"""
    # Get first order per customer
    first_orders = delivery.loc[delivery.groupby('customer_id')['order_date'].idxmin()]

    # Calculate percentage
    immediate = (first_orders['order_date'] == first_orders['customer_pref_delivery_date']).sum()
    total = len(first_orders)
    percentage = round(immediate / total * 100, 2) if total > 0 else 0

    return pd.DataFrame({'immediate_percentage': [percentage]})


# ============================================================================
# MEDIUM PROBLEMS - Additional Solutions
# ============================================================================

def user_activity_30_days(activity: pd.DataFrame) -> pd.DataFrame:
    """Count active users in past 30 days"""
    end_date = pd.Timestamp('2019-07-27')
    start_date = end_date - pd.Timedelta(days=29)

    filtered = activity[
        (activity['activity_date'] >= start_date) &
        (activity['activity_date'] <= end_date)
    ]

    result = filtered.groupby('activity_date')['user_id'].nunique().reset_index()
    result.columns = ['day', 'active_users']

    return result


def sales_analysis_iii(product: pd.DataFrame, sales: pd.DataFrame) -> pd.DataFrame:
    """Find products sold only in Q1 2019"""
    q1_start = pd.Timestamp('2019-01-01')
    q1_end = pd.Timestamp('2019-03-31')

    # Products sold in Q1
    q1_products = sales[
        (sales['sale_date'] >= q1_start) &
        (sales['sale_date'] <= q1_end)
    ]['product_id'].unique()

    # Products sold outside Q1
    outside_q1 = sales[
        (sales['sale_date'] < q1_start) |
        (sales['sale_date'] > q1_end)
    ]['product_id'].unique()

    # Products only in Q1
    only_q1 = set(q1_products) - set(outside_q1)

    return product[product['product_id'].isin(only_q1)][['product_id', 'product_name']]


def find_followers(followers: pd.DataFrame) -> pd.DataFrame:
    """Count followers per user"""
    result = followers.groupby('user_id').size().reset_index(name='followers_count')
    return result.sort_values('user_id')


def biggest_single_number(my_numbers: pd.DataFrame) -> pd.DataFrame:
    """Find biggest number that appears only once"""
    counts = my_numbers['num'].value_counts()
    single_numbers = counts[counts == 1].index

    if len(single_numbers) == 0:
        return pd.DataFrame({'num': [None]})

    return pd.DataFrame({'num': [single_numbers.max()]})


def all_products_customers(customer: pd.DataFrame, product: pd.DataFrame) -> pd.DataFrame:
    """Find customers who bought all products"""
    total_products = len(product)

    customer_products = customer.groupby('customer_id')['product_key'].nunique()
    result = customer_products[customer_products == total_products].reset_index()

    return result[['customer_id']]


def employees_with_reports(employees: pd.DataFrame) -> pd.DataFrame:
    """Find employees who have direct reports"""
    reports = employees[employees['reports_to'].notna()]

    result = reports.groupby('reports_to').agg(
        reports_count=('employee_id', 'count'),
        average_age=('age', 'mean')
    ).reset_index()

    result['average_age'] = result['average_age'].round()
    result = result.merge(
        employees[['employee_id', 'name']],
        left_on='reports_to',
        right_on='employee_id'
    )

    return result[['employee_id', 'name', 'reports_count', 'average_age']].sort_values('employee_id')


def triangle_judgement(triangle: pd.DataFrame) -> pd.DataFrame:
    """Determine if three sides form a valid triangle"""
    triangle['triangle'] = triangle.apply(
        lambda row: 'Yes' if (
            row['x'] + row['y'] > row['z'] and
            row['x'] + row['z'] > row['y'] and
            row['y'] + row['z'] > row['x']
        ) else 'No',
        axis=1
    )
    return triangle


def last_person_in_bus(queue: pd.DataFrame) -> pd.DataFrame:
    """Find last person to fit in the bus (max 1000kg)"""
    queue = queue.sort_values('turn')
    queue['cumulative_weight'] = queue['weight'].cumsum()

    result = queue[queue['cumulative_weight'] <= 1000]

    if result.empty:
        return pd.DataFrame({'person_name': []})

    return pd.DataFrame({'person_name': [result.iloc[-1]['person_name']]})


def count_salary_categories(accounts: pd.DataFrame) -> pd.DataFrame:
    """Count accounts in each salary category"""
    low = (accounts['income'] < 20000).sum()
    average = ((accounts['income'] >= 20000) & (accounts['income'] <= 50000)).sum()
    high = (accounts['income'] > 50000).sum()

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ============================================================================
# EASY PROBLEMS - Python Solutions
# ============================================================================

def recyclable_and_low_fat(products: pd.DataFrame) -> pd.DataFrame:
    """Select recyclable and low fat products"""
    return products[
        (products['low_fats'] == 'Y') &
        (products['recyclable'] == 'Y')
    ][['product_id']]


def find_customer_referee(customer: pd.DataFrame) -> pd.DataFrame:
    """Find customers not referred by customer 2"""
    return customer[
        customer['referee_id'].isna() |
        (customer['referee_id'] != 2)
    ][['name']]


def big_countries(world: pd.DataFrame) -> pd.DataFrame:
    """Find big countries (area >= 3M or population >= 25M)"""
    return world[
        (world['area'] >= 3000000) |
        (world['population'] >= 25000000)
    ][['name', 'population', 'area']]


def article_views(views: pd.DataFrame) -> pd.DataFrame:
    """Find authors who viewed their own articles"""
    result = views[views['author_id'] == views['viewer_id']][['author_id']]
    result = result.drop_duplicates().rename(columns={'author_id': 'id'})
    return result.sort_values('id')


def invalid_tweets(tweets: pd.DataFrame) -> pd.DataFrame:
    """Find tweets with content length > 15"""
    return tweets[tweets['content'].str.len() > 15][['tweet_id']]


def replace_employee_id(employees: pd.DataFrame, employee_uni: pd.DataFrame) -> pd.DataFrame:
    """Replace employee ID with unique identifier"""
    return employees.merge(
        employee_uni,
        left_on='id',
        right_on='id',
        how='left'
    )[['unique_id', 'name']]


def sales_analysis(sales: pd.DataFrame, product: pd.DataFrame) -> pd.DataFrame:
    """Product sales analysis"""
    return sales.merge(product, on='product_id')[['product_name', 'year', 'price']]


def no_transactions(visits: pd.DataFrame, transactions: pd.DataFrame) -> pd.DataFrame:
    """Customers who visited but made no transactions"""
    merged = visits.merge(transactions, on='visit_id', how='left')
    no_trans = merged[merged['transaction_id'].isna()]
    result = no_trans.groupby('customer_id').size().reset_index(name='count_no_trans')
    return result


def rising_temperature(weather: pd.DataFrame) -> pd.DataFrame:
    """Find days with temperature higher than previous day"""
    weather = weather.sort_values('recordDate')
    weather['prev_temp'] = weather['temperature'].shift(1)
    weather['prev_date'] = weather['recordDate'].shift(1)
    weather['date_diff'] = (weather['recordDate'] - weather['prev_date']).dt.days

    result = weather[
        (weather['temperature'] > weather['prev_temp']) &
        (weather['date_diff'] == 1)
    ][['id']]

    return result


def average_time_process(activity: pd.DataFrame) -> pd.DataFrame:
    """Calculate average processing time per machine"""
    # Pivot to get start and end times
    start = activity[activity['activity_type'] == 'start'][['machine_id', 'process_id', 'timestamp']]
    end = activity[activity['activity_type'] == 'end'][['machine_id', 'process_id', 'timestamp']]

    merged = start.merge(
        end,
        on=['machine_id', 'process_id'],
        suffixes=('_start', '_end')
    )
    merged['processing_time'] = merged['timestamp_end'] - merged['timestamp_start']

    result = merged.groupby('machine_id')['processing_time'].mean().reset_index()
    result['processing_time'] = result['processing_time'].round(3)

    return result


def employee_bonus(employee: pd.DataFrame, bonus: pd.DataFrame) -> pd.DataFrame:
    """Find employees with bonus < 1000 or no bonus"""
    merged = employee.merge(bonus, on='empId', how='left')
    result = merged[(merged['bonus'] < 1000) | merged['bonus'].isna()]
    return result[['name', 'bonus']]


def students_and_examinations(students: pd.DataFrame, subjects: pd.DataFrame,
                              examinations: pd.DataFrame) -> pd.DataFrame:
    """Count exam attendance per student and subject"""
    # Cross join students and subjects
    cross = students.merge(subjects, how='cross')

    # Count examinations
    exam_counts = examinations.groupby(['student_id', 'subject_name']).size().reset_index(name='attended_exams')

    # Merge with cross product
    result = cross.merge(
        exam_counts,
        on=['student_id', 'subject_name'],
        how='left'
    )
    result['attended_exams'] = result['attended_exams'].fillna(0).astype(int)

    return result[['student_id', 'student_name', 'subject_name', 'attended_exams']].sort_values(
        ['student_id', 'subject_name']
    )


def managers_with_five_reports(employee: pd.DataFrame) -> pd.DataFrame:
    """Find managers with at least 5 direct reports"""
    reports = employee['managerId'].value_counts()
    manager_ids = reports[reports >= 5].index

    return employee[employee['id'].isin(manager_ids)][['name']]


def confirmation_rate(signups: pd.DataFrame, confirmations: pd.DataFrame) -> pd.DataFrame:
    """Calculate confirmation rate per user"""
    # Count confirmations per user
    conf_stats = confirmations.groupby('user_id').agg(
        total_actions=('action', 'count'),
        confirmed_actions=('action', lambda x: (x == 'confirmed').sum())
    ).reset_index()

    # Merge with signups
    result = signups.merge(conf_stats, on='user_id', how='left')
    result['total_actions'] = result['total_actions'].fillna(0)
    result['confirmed_actions'] = result['confirmed_actions'].fillna(0)

    # Calculate rate
    result['confirmation_rate'] = (
        result['confirmed_actions'] / result['total_actions'].replace(0, 1)
    ).fillna(0).round(2)

    return result[['user_id', 'confirmation_rate']]


def not_boring_movies(cinema: pd.DataFrame) -> pd.DataFrame:
    """Find not boring movies with odd ID"""
    result = cinema[
        (cinema['id'] % 2 == 1) &
        (cinema['description'] != 'boring')
    ]
    return result.sort_values('rating', ascending=False)


def average_selling_price(prices: pd.DataFrame, units_sold: pd.DataFrame) -> pd.DataFrame:
    """Calculate average selling price"""
    # Merge on product and date range
    merged = prices.merge(units_sold, on='product_id', how='left')
    merged = merged[
        (merged['purchase_date'].isna()) |
        ((merged['purchase_date'] >= merged['start_date']) &
         (merged['purchase_date'] <= merged['end_date']))
    ]

    # Calculate weighted average
    merged['total_value'] = merged['units'].fillna(0) * merged['price']

    result = merged.groupby('product_id').agg(
        total_value=('total_value', 'sum'),
        total_units=('units', 'sum')
    ).reset_index()

    result['average_price'] = (
        result['total_value'] / result['total_units'].replace(0, 1)
    ).fillna(0).round(2)

    return result[['product_id', 'average_price']]


def project_employees(project: pd.DataFrame, employee: pd.DataFrame) -> pd.DataFrame:
    """Calculate average years of experience per project"""
    merged = project.merge(employee, on='employee_id')
    result = merged.groupby('project_id')['experience_years'].mean().reset_index()
    result['experience_years'] = result['experience_years'].round(2)
    result.columns = ['project_id', 'average_years']
    return result


def contest_attendance(users: pd.DataFrame, register: pd.DataFrame) -> pd.DataFrame:
    """Calculate percentage of users attending each contest"""
    total_users = len(users)

    result = register.groupby('contest_id')['user_id'].nunique().reset_index()
    result['percentage'] = (result['user_id'] / total_users * 100).round(2)
    result = result.rename(columns={'user_id': 'registered_users'})

    return result[['contest_id', 'percentage']].sort_values(
        ['percentage', 'contest_id'],
        ascending=[False, True]
    )


def queries_quality(queries: pd.DataFrame) -> pd.DataFrame:
    """Calculate query quality and poor query percentage"""
    queries = queries[queries['query_name'].notna()]

    # Calculate quality
    queries['quality_score'] = queries['rating'] / queries['position']
    queries['is_poor'] = (queries['rating'] < 3).astype(int)

    result = queries.groupby('query_name').agg(
        quality=('quality_score', 'mean'),
        poor_query_percentage=('is_poor', lambda x: x.mean() * 100)
    ).reset_index()

    result['quality'] = result['quality'].round(2)
    result['poor_query_percentage'] = result['poor_query_percentage'].round(2)

    return result


def monthly_transactions(transactions: pd.DataFrame) -> pd.DataFrame:
    """Calculate monthly transaction statistics"""
    transactions['month'] = transactions['trans_date'].dt.to_period('M').astype(str)

    result = transactions.groupby(['month', 'country']).agg(
        trans_count=('id', 'count'),
        approved_count=('state', lambda x: (x == 'approved').sum()),
        trans_total_amount=('amount', 'sum'),
        approved_total_amount=('amount', lambda x: x[transactions.loc[x.index, 'state'] == 'approved'].sum())
    ).reset_index()

    return result


def immediate_food_delivery(delivery: pd.DataFrame) -> pd.DataFrame:
    """Calculate percentage of immediate first orders"""
    # Get first order per customer
    first_orders = delivery.loc[delivery.groupby('customer_id')['order_date'].idxmin()]

    # Calculate percentage
    immediate = (first_orders['order_date'] == first_orders['customer_pref_delivery_date']).sum()
    total = len(first_orders)
    percentage = round(immediate / total * 100, 2) if total > 0 else 0

    return pd.DataFrame({'immediate_percentage': [percentage]})


# ============================================================================
# MEDIUM PROBLEMS - Additional Solutions
# ============================================================================

def user_activity_30_days(activity: pd.DataFrame) -> pd.DataFrame:
    """Count active users in past 30 days"""
    end_date = pd.Timestamp('2019-07-27')
    start_date = end_date - pd.Timedelta(days=29)

    filtered = activity[
        (activity['activity_date'] >= start_date) &
        (activity['activity_date'] <= end_date)
    ]

    result = filtered.groupby('activity_date')['user_id'].nunique().reset_index()
    result.columns = ['day', 'active_users']

    return result


def sales_analysis_iii(product: pd.DataFrame, sales: pd.DataFrame) -> pd.DataFrame:
    """Find products sold only in Q1 2019"""
    q1_start = pd.Timestamp('2019-01-01')
    q1_end = pd.Timestamp('2019-03-31')

    # Products sold in Q1
    q1_products = sales[
        (sales['sale_date'] >= q1_start) &
        (sales['sale_date'] <= q1_end)
    ]['product_id'].unique()

    # Products sold outside Q1
    outside_q1 = sales[
        (sales['sale_date'] < q1_start) |
        (sales['sale_date'] > q1_end)
    ]['product_id'].unique()

    # Products only in Q1
    only_q1 = set(q1_products) - set(outside_q1)

    return product[product['product_id'].isin(only_q1)][['product_id', 'product_name']]


def find_followers(followers: pd.DataFrame) -> pd.DataFrame:
    """Count followers per user"""
    result = followers.groupby('user_id').size().reset_index(name='followers_count')
    return result.sort_values('user_id')


def biggest_single_number(my_numbers: pd.DataFrame) -> pd.DataFrame:
    """Find biggest number that appears only once"""
    counts = my_numbers['num'].value_counts()
    single_numbers = counts[counts == 1].index

    if len(single_numbers) == 0:
        return pd.DataFrame({'num': [None]})

    return pd.DataFrame({'num': [single_numbers.max()]})


def all_products_customers(customer: pd.DataFrame, product: pd.DataFrame) -> pd.DataFrame:
    """Find customers who bought all products"""
    total_products = len(product)

    customer_products = customer.groupby('customer_id')['product_key'].nunique()
    result = customer_products[customer_products == total_products].reset_index()

    return result[['customer_id']]


def employees_with_reports(employees: pd.DataFrame) -> pd.DataFrame:
    """Find employees who have direct reports"""
    reports = employees[employees['reports_to'].notna()]

    result = reports.groupby('reports_to').agg(
        reports_count=('employee_id', 'count'),
        average_age=('age', 'mean')
    ).reset_index()

    result['average_age'] = result['average_age'].round()
    result = result.merge(
        employees[['employee_id', 'name']],
        left_on='reports_to',
        right_on='employee_id'
    )

    return result[['employee_id', 'name', 'reports_count', 'average_age']].sort_values('employee_id')


def triangle_judgement(triangle: pd.DataFrame) -> pd.DataFrame:
    """Determine if three sides form a valid triangle"""
    triangle['triangle'] = triangle.apply(
        lambda row: 'Yes' if (
            row['x'] + row['y'] > row['z'] and
            row['x'] + row['z'] > row['y'] and
            row['y'] + row['z'] > row['x']
        ) else 'No',
        axis=1
    )
    return triangle


def last_person_in_bus(queue: pd.DataFrame) -> pd.DataFrame:
    """Find last person to fit in the bus (max 1000kg)"""
    queue = queue.sort_values('turn')
    queue['cumulative_weight'] = queue['weight'].cumsum()

    result = queue[queue['cumulative_weight'] <= 1000]

    if result.empty:
        return pd.DataFrame({'person_name': []})

    return pd.DataFrame({'person_name': [result.iloc[-1]['person_name']]})


def count_salary_categories(accounts: pd.DataFrame) -> pd.DataFrame:
    """Count accounts in each salary category"""
    low = (accounts['income'] < 20000).sum()
    average = ((accounts['income'] >= 20000) & (accounts['income'] <= 50000)).sum()
    high = (accounts['income'] > 50000).sum()

    return pd.DataFrame({
        'category': ['Low Salary', 'Average Salary', 'High Salary'],
        'accounts_count': [low, average, high]
    })


def fix_names(users: pd.DataFrame) -> pd.DataFrame:
    """Fix user names to proper case"""
    users['name'] = users['name'].str.capitalize()
    return users.sort_values('user_id')


def patients_with_condition(patients: pd.DataFrame) -> pd.DataFrame:
    """Find patients with DIAB1 condition"""
    result = patients[
        patients['conditions'].str.contains(r'\bDIAB1', regex=True, na=False)
    ]
    return result


def second_highest_salary(employee: pd.DataFrame) -> pd.DataFrame:
    """Find second highest salary"""
    unique_salaries = employee['salary'].drop_duplicates().sort_values(ascending=False)

    if len(unique_salaries) < 2:
        return pd.DataFrame({'SecondHighestSalary': [None]})

    return pd.DataFrame({'SecondHighestSalary': [unique_salaries.iloc[1]]})


def group_sold_products(activities: pd.DataFrame) -> pd.DataFrame:
    """Group products sold by date"""
    result = activities.groupby('sell_date')['product'].apply(
        lambda x: ','.join(sorted(x.unique()))
    ).reset_index()

    num_sold = activities.groupby('sell_date')['product'].nunique().reset_index()
    num_sold.columns = ['sell_date', 'num_sold']

    result = result.merge(num_sold, on='sell_date')
    result.columns = ['sell_date', 'products', 'num_sold']

    return result[['sell_date', 'num_sold', 'products']].sort_values('sell_date')


def products_ordered_feb_2020(products: pd.DataFrame, orders: pd.DataFrame) -> pd.DataFrame:
    """List products ordered in February 2020 with unit >= 100"""
    feb_orders = orders[
        (orders['order_date'].dt.year == 2020) &
        (orders['order_date'].dt.month == 2)
    ]

    result = feb_orders.groupby('product_id')['unit'].sum().reset_index()
    result = result[result['unit'] >= 100]

    result = result.merge(products, on='product_id')
    return result[['product_name', 'unit']]


def valid_emails(users: pd.DataFrame) -> pd.DataFrame:
    """Find users with valid LeetCode emails"""
    pattern = r'^[A-Za-z][A-Za-z0-9_.-]*@leetcode\.com
    result = users[users['mail'].str.match(pattern, na=False)]
    return result


def top_travellers(users: pd.DataFrame, rides: pd.DataFrame) -> pd.DataFrame:
    """Calculate total distance travelled per user"""
    travelled = rides.groupby('user_id')['distance'].sum().reset_index()
    travelled.columns = ['id', 'travelled_distance']

    result = users.merge(travelled, on='id', how='left')
    result['travelled_distance'] = result['travelled_distance'].fillna(0)

    return result[['name', 'travelled_distance']].sort_values(
        ['travelled_distance', 'name'],
        ascending=[False, True]
    )


def market_analysis(users: pd.DataFrame, orders: pd.DataFrame, items: pd.DataFrame) -> pd.DataFrame:
    """Count orders per user in 2019"""
    orders_2019 = orders[orders['order_date'].dt.year == 2019]
    order_counts = orders_2019.groupby('buyer_id').size().reset_index(name='orders_in_2019')

    result = users.merge(order_counts, left_on='user_id', right_on='buyer_id', how='left')
    result['orders_in_2019'] = result['orders_in_2019'].fillna(0).astype(int)

    return result[['user_id', 'join_date', 'orders_in_2019']].rename(
        columns={'user_id': 'buyer_id'}
    )


def sales_person(sales_person: pd.DataFrame, company: pd.DataFrame, orders: pd.DataFrame) -> pd.DataFrame:
    """Find sales persons with no orders from RED company"""
    red_orders = orders.merge(company, on='com_id')
    red_orders = red_orders[red_orders['name'] == 'RED']
    red_sales_ids = red_orders['sales_id'].unique()

    result = sales_person[~sales_person['sales_id'].isin(red_sales_ids)]
    return result[['name']]


def tree_node(tree: pd.DataFrame) -> pd.DataFrame:
    """Classify tree nodes as Root, Inner, or Leaf"""
    def get_type(row):
        if pd.isna(row['p_id']):
            return 'Root'
        elif row['id'] in tree['p_id'].values:
            return 'Inner'
        else:
            return 'Leaf'

    tree['type'] = tree.apply(get_type, axis=1)
    return tree[['id', 'type']]


def total_time_employee(employees: pd.DataFrame) -> pd.DataFrame:
    """Calculate total time spent per employee per day"""
    employees['total_time'] = employees['out_time'] - employees['in_time']

    result = employees.groupby(['event_day', 'emp_id'])['total_time'].sum().reset_index()
    result.columns = ['day', 'emp_id', 'total_time']

    return result


def special_bonus(employees: pd.DataFrame) -> pd.DataFrame:
    """Calculate special bonus for employees"""
    employees['bonus'] = employees.apply(
        lambda row: row['salary'] if row['employee_id'] % 2 == 1 and not row['name'].startswith('M') else 0,
        axis=1
    )
    return employees[['employee_id', 'bonus']].sort_values('employee_id')


def actors_and_directors(actor_director: pd.DataFrame) -> pd.DataFrame:
    """Find actor-director pairs who cooperated at least 3 times"""
    cooperations = actor_director.groupby(['actor_id', 'director_id']).size().reset_index(name='count')
    result = cooperations[cooperations['count'] >= 3]
    return result[['actor_id', 'director_id']]


def employees_missing_info(employees: pd.DataFrame, salaries: pd.DataFrame) -> pd.DataFrame:
    """Find employees with missing information"""
    all_employees = set(employees['employee_id']).union(set(salaries['employee_id']))
    complete_info = set(employees['employee_id']).intersection(set(salaries['employee_id']))

    missing = all_employees - complete_info
    result = pd.DataFrame({'employee_id': sorted(missing)})
    return result


def earning_more_than_managers(employee: pd.DataFrame) -> pd.DataFrame:
    """Find employees earning more than their managers"""
    merged = employee.merge(
        employee[['id', 'salary']],
        left_on='managerId',
        right_on='id',
        suffixes=('', '_manager')
    )

    result = merged[merged['salary'] > merged['salary_manager']]
    return result[['name']].rename(columns={'name': 'Employee'})


def duplicate_emails(person: pd.DataFrame) -> pd.DataFrame:
    """Find duplicate emails"""
    email_counts = person['email'].value_counts()
    duplicates = email_counts[email_counts > 1].index
    return pd.DataFrame({'email': duplicates})


def customers_never_order(customers: pd.DataFrame, orders: pd.DataFrame) -> pd.DataFrame:
    """Find customers who never ordered"""
    ordered_customers = orders['customerId'].unique()
    result = customers[~customers['id'].isin(ordered_customers)]
    return result[['name']].rename(columns={'name': 'Customers'})


def bank_account_summary(users: pd.DataFrame, transactions: pd.DataFrame) -> pd.DataFrame:
    """Find users with balance > 10000"""
    balance = transactions.groupby('account')['amount'].sum().reset_index(name='balance')
    result = balance[balance['balance'] > 10000]

    result = result.merge(users, on='account')
    return result[['name', 'balance']]


def game_play_first_login(activity: pd.DataFrame) -> pd.DataFrame:
    """Find first login date for each player"""
    result = activity.groupby('player_id')['event_date'].min().reset_index()
    result.columns = ['player_id', 'first_login']
    return result


def trips_and_users(trips: pd.DataFrame, users: pd.DataFrame) -> pd.DataFrame:
    """Calculate cancellation rate for trips"""
    # Filter out banned users
    valid_users = users[users['banned'] == 'No']['users_id']

    valid_trips = trips[
        trips['client_id'].isin(valid_users) &
        trips['driver_id'].isin(valid_users)
    ]

    # Filter by date range
    valid_trips = valid_trips[
        (valid_trips['request_at'] >= '2013-10-01') &
        (valid_trips['request_at'] <= '2013-10-03')
    ]

    # Calculate cancellation rate
    result = valid_trips.groupby('request_at').apply(
        lambda x: (x['status'].str.startswith('cancelled')).sum() / len(x)
    ).reset_index()

    result.columns = ['Day', 'Cancellation Rate']
    result['Cancellation Rate'] = result['Cancellation Rate'].round(2)

    return result


def reformat_department_table(department: pd.DataFrame) -> pd.DataFrame:
    """Reformat department revenue table"""
    result = department.pivot_table(
        index='id',
        columns='month',
        values='revenue',
        aggfunc='sum'
    ).reset_index()

    # Rename columns
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    new_columns = ['id'] + [f'{m}_Revenue' for m in months if m in result.columns]

    result.columns = ['id'] + [f'{col}_Revenue' for col in result.columns[1:]]
    return result


def daily_leads_partners(daily_sales: pd.DataFrame) -> pd.DataFrame:
    """Count unique leads and partners per date and make"""
    result = daily_sales.groupby(['date_id', 'make_name']).agg(
        unique_leads=('lead_id', 'nunique'),
        unique_partners=('partner_id', 'nunique')
    ).reset_index()

    return result


def combine_two_tables(person: pd.DataFrame, address: pd.DataFrame) -> pd.DataFrame:
    """Combine person and address tables"""
    result = person.merge(address, on='personId', how='left')
    return result[['firstName', 'lastName', 'city', 'state']]


def latest_login_2020(logins: pd.DataFrame) -> pd.DataFrame:
    """Find latest login in 2020 for each user"""
    logins_2020 = logins[logins['time_stamp'].dt.year == 2020]
    result = logins_2020.groupby('user_id')['time_stamp'].max().reset_index()
    result.columns = ['user_id', 'last_stamp']
    return result


def rearrange_products(products: pd.DataFrame) -> pd.DataFrame:
    """Rearrange products table from wide to long format"""
    result = pd.melt(
        products,
        id_vars=['product_id'],
        value_vars=['store1', 'store2', 'store3'],
        var_name='store',
        value_name='price'
    )

    result = result[result['price'].notna()]
    return result


def friend_requests_most_friends(request_accepted: pd.DataFrame) -> pd.DataFrame:
    """Find person with most friends"""
    # Combine requester and accepter
    friends = pd.concat([
        request_accepted[['requester_id']].rename(columns={'requester_id': 'id'}),
        request_accepted[['accepter_id']].rename(columns={'accepter_id': 'id'})
    ])

    friend_counts = friends.groupby('id').size().reset_index(name='num')
    result = friend_counts.sort_values('num', ascending=False).head(1)

    return result


def investments_2016(insurance: pd.DataFrame) -> pd.DataFrame:
    """Calculate total investment for specific conditions"""
    # Find tiv_2015 values that appear more than once
    tiv_2015_counts = insurance['tiv_2015'].value_counts()
    valid_tiv = tiv_2015_counts[tiv_2015_counts > 1].index

    # Find unique locations
    location_counts = insurance.groupby(['lat', 'lon']).size()
    unique_locations = location_counts[location_counts == 1].index

    # Filter insurance records
    result = insurance[
        insurance['tiv_2015'].isin(valid_tiv) &
        insurance.apply(lambda row: (row['lat'], row['lon']) in unique_locations, axis=1)
    ]

    total = result['tiv_2016'].sum()
    return pd.DataFrame({'tiv_2016': [round(total, 2)]})


def restaurant_growth(customer: pd.DataFrame) -> pd.DataFrame:
    """Calculate 7-day moving average for restaurant"""
    daily_amount = customer.groupby('visited_on')['amount'].sum().reset_index()
    daily_amount = daily_amount.sort_values('visited_on')

    # Calculate rolling sum
    daily_amount['amount'] = daily_amount['amount'].rolling(window=7, min_periods=7).sum()
    daily_amount['average_amount'] = (daily_amount['amount'] / 7).round(2)

    # Filter rows with at least 7 days
    result = daily_amount[daily_amount['amount'].notna()].copy()
    result['amount'] = result['amount'].astype(int)

    return result


# ============================================================================
# Example Usage and Testing
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("LeetCode SQL Problems - Python Solutions")
    print("="*60)

    # Test Recyclable Products
    print("\n1. Recyclable and Low Fat Products:")
    products = pd.DataFrame({
        'product_id': [1, 2, 3, 4],
        'low_fats': ['Y', 'Y', 'N', 'Y'],
        'recyclable': ['Y', 'N', 'Y', 'Y']
    })
    print(recyclable_and_low_fat(products))

    # Test Big Countries
    print("\n2. Big Countries:")
    world = pd.DataFrame({
        'name': ['USA', 'Canada', 'China', 'Monaco'],
        'population': [330000000, 38000000, 1400000000, 40000],
        'area': [9834000, 9985000, 9597000, 2]
    })
    print(big_countries(world))

    # Test Rising Temperature
    print("\n3. Rising Temperature:")
    weather = pd.DataFrame({
        'id': [1, 2, 3, 4],
        'recordDate': pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04']),
        'temperature': [10, 12, 8, 15]
    })
    print(rising_temperature(weather))

    print("\n" + "="*60)
    print("All solutions ready for use!")
    print("="*60)

In [ ]:
-- ============================================================================
-- COMPREHENSIVE PL/SQL SOLUTIONS FOR LEETCODE SQL PROBLEMS
-- ============================================================================

-- ============================================================================
-- UTILITY FUNCTIONS AND TYPES
-- ============================================================================

-- Create a package for common utilities
CREATE OR REPLACE PACKAGE leetcode_utils AS
    TYPE num_array IS TABLE OF NUMBER;
    TYPE varchar_array IS TABLE OF VARCHAR2(1000);
    TYPE date_array IS TABLE OF DATE;

    PROCEDURE log_message(p_message VARCHAR2);
    FUNCTION get_nth_value(p_array num_array, p_position NUMBER) RETURN NUMBER;
END leetcode_utils;
/

CREATE OR REPLACE PACKAGE BODY leetcode_utils AS
    PROCEDURE log_message(p_message VARCHAR2) IS
    BEGIN
        DBMS_OUTPUT.PUT_LINE(TO_CHAR(SYSTIMESTAMP, 'YYYY-MM-DD HH24:MI:SS') || ' - ' || p_message);
    END log_message;

    FUNCTION get_nth_value(p_array num_array, p_position NUMBER) RETURN NUMBER IS
    BEGIN
        IF p_position <= 0 OR p_position > p_array.COUNT THEN
            RETURN NULL;
        END IF;
        RETURN p_array(p_position);
    END get_nth_value;
END leetcode_utils;
/

-- ============================================================================
-- 177. Nth Highest Salary - Multiple Implementations
-- ============================================================================

-- Function version 1: Simple approach
CREATE OR REPLACE FUNCTION get_nth_highest_salary_v1(p_n IN NUMBER)
RETURN NUMBER
DETERMINISTIC
IS
    v_salary NUMBER;
BEGIN
    SELECT salary INTO v_salary
    FROM (
        SELECT DISTINCT salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) as rank
        FROM Employee
    )
    WHERE rank = p_n;

    RETURN v_salary;
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RETURN NULL;
    WHEN OTHERS THEN
        leetcode_utils.log_message('Error in get_nth_highest_salary_v1: ' || SQLERRM);
        RETURN NULL;
END get_nth_highest_salary_v1;
/

-- Function version 2: With validation
CREATE OR REPLACE FUNCTION get_nth_highest_salary_v2(p_n IN NUMBER)
RETURN NUMBER
IS
    v_salary NUMBER;
    v_max_rank NUMBER;
BEGIN
    -- Validate input
    IF p_n IS NULL OR p_n <= 0 THEN
        RETURN NULL;
    END IF;

    -- Check if N exists
    SELECT MAX(rank) INTO v_max_rank
    FROM (
        SELECT DISTINCT salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) as rank
        FROM Employee
    );

    IF p_n > v_max_rank THEN
        RETURN NULL;
    END IF;

    -- Get the salary
    SELECT salary INTO v_salary
    FROM (
        SELECT DISTINCT salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) as rank
        FROM Employee
    )
    WHERE rank = p_n;

    RETURN v_salary;
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RETURN NULL;
    WHEN OTHERS THEN
        RAISE;
END get_nth_highest_salary_v2;
/

-- Procedure version with OUT parameter
CREATE OR REPLACE PROCEDURE get_nth_salary_proc(
    p_n IN NUMBER,
    p_salary OUT NUMBER,
    p_status OUT VARCHAR2
) IS
BEGIN
    p_status := 'SUCCESS';

    SELECT salary INTO p_salary
    FROM (
        SELECT DISTINCT salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) as rn
        FROM Employee
    )
    WHERE rn = p_n;

EXCEPTION
    WHEN NO_DATA_FOUND THEN
        p_salary := NULL;
        p_status := 'NOT_FOUND';
    WHEN OTHERS THEN
        p_salary := NULL;
        p_status := 'ERROR: ' || SQLERRM;
END get_nth_salary_proc;
/

-- ============================================================================
-- 178. Rank Scores - Procedure with Output
-- ============================================================================

CREATE OR REPLACE PROCEDURE rank_all_scores(
    p_cursor OUT SYS_REFCURSOR
) IS
BEGIN
    OPEN p_cursor FOR
        SELECT score,
               DENSE_RANK() OVER (ORDER BY score DESC) as rank
        FROM Scores
        ORDER BY score DESC;
END rank_all_scores;
/

-- Procedure to materialize ranked scores
CREATE OR REPLACE PROCEDURE materialize_ranked_scores IS
    v_count NUMBER := 0;
BEGIN
    -- Create or replace temp table
    EXECUTE IMMEDIATE 'TRUNCATE TABLE Scores_Ranked';

    INSERT INTO Scores_Ranked (score_id, score, rank)
    SELECT id, score,
           DENSE_RANK() OVER (ORDER BY score DESC) as rank
    FROM Scores;

    v_count := SQL%ROWCOUNT;
    COMMIT;

    leetcode_utils.log_message('Ranked ' || v_count || ' scores');
EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        leetcode_utils.log_message('Error: ' || SQLERRM);
        RAISE;
END materialize_ranked_scores;
/

-- ============================================================================
-- 180. Consecutive Numbers - Advanced Implementation
-- ============================================================================

CREATE OR REPLACE FUNCTION find_consecutive_numbers(p_count IN NUMBER DEFAULT 3)
RETURN SYS_REFCURSOR
IS
    v_cursor SYS_REFCURSOR;
BEGIN
    OPEN v_cursor FOR
        WITH numbered_logs AS (
            SELECT num,
                   LAG(num, 1) OVER (ORDER BY id) as prev1,
                   LAG(num, 2) OVER (ORDER BY id) as prev2,
                   LEAD(num, 1) OVER (ORDER BY id) as next1,
                   LEAD(num, 2) OVER (ORDER BY id) as next2
            FROM Logs
        )
        SELECT DISTINCT num as ConsecutiveNums
        FROM numbered_logs
        WHERE (num = prev1 AND num = prev2) OR
              (num = prev1 AND num = next1) OR
              (num = next1 AND num = next2);

    RETURN v_cursor;
END find_consecutive_numbers;
/

-- Procedure with collection output
CREATE OR REPLACE PROCEDURE get_consecutive_numbers(
    p_min_consecutive IN NUMBER DEFAULT 3,
    p_numbers OUT leetcode_utils.num_array
) IS
    CURSOR c_logs IS
        SELECT id, num FROM Logs ORDER BY id;

    TYPE t_log_rec IS RECORD (
        id NUMBER,
        num NUMBER
    );
    TYPE t_log_table IS TABLE OF t_log_rec;

    v_logs t_log_table;
    v_consecutive_count NUMBER;
    v_current_num NUMBER;
    v_result_set leetcode_utils.num_array := leetcode_utils.num_array();
BEGIN
    -- Load all logs
    SELECT id, num
    BULK COLLECT INTO v_logs
    FROM Logs
    ORDER BY id;

    -- Find consecutive numbers
    FOR i IN 1..v_logs.COUNT - (p_min_consecutive - 1) LOOP
        v_current_num := v_logs(i).num;
        v_consecutive_count := 1;

        FOR j IN i+1..LEAST(i + p_min_consecutive - 1, v_logs.COUNT) LOOP
            IF v_logs(j).num = v_current_num THEN
                v_consecutive_count := v_consecutive_count + 1;
            ELSE
                EXIT;
            END IF;
        END LOOP;

        IF v_consecutive_count >= p_min_consecutive THEN
            -- Check if already in result
            IF NOT v_result_set.EXISTS(1) OR v_current_num NOT MEMBER OF v_result_set THEN
                v_result_set.EXTEND;
                v_result_set(v_result_set.LAST) := v_current_num;
            END IF;
        END IF;
    END LOOP;

    p_numbers := v_result_set;
END get_consecutive_numbers;
/

-- ============================================================================
-- 184 & 185. Department Salaries - Package
-- ============================================================================

CREATE OR REPLACE PACKAGE dept_salary_pkg AS
    PROCEDURE get_highest_salaries(p_cursor OUT SYS_REFCURSOR);
    PROCEDURE get_top_n_salaries(p_n IN NUMBER, p_cursor OUT SYS_REFCURSOR);
    FUNCTION get_dept_top_salary(p_dept_id IN NUMBER) RETURN NUMBER;
END dept_salary_pkg;
/

CREATE OR REPLACE PACKAGE BODY dept_salary_pkg AS

    PROCEDURE get_highest_salaries(p_cursor OUT SYS_REFCURSOR) IS
    BEGIN
        OPEN p_cursor FOR
            SELECT d.name as Department,
                   e.name as Employee,
                   e.salary as Salary
            FROM Employee e
            JOIN Department d ON e.departmentId = d.id
            WHERE (e.departmentId, e.salary) IN (
                SELECT departmentId, MAX(salary)
                FROM Employee
                GROUP BY departmentId
            )
            ORDER BY d.name, e.name;
    END get_highest_salaries;

    PROCEDURE get_top_n_salaries(p_n IN NUMBER, p_cursor OUT SYS_REFCURSOR) IS
    BEGIN
        OPEN p_cursor FOR
            SELECT Department, Employee, Salary
            FROM (
                SELECT d.name as Department,
                       e.name as Employee,
                       e.salary as Salary,
                       DENSE_RANK() OVER (
                           PARTITION BY d.id
                           ORDER BY e.salary DESC
                       ) as salary_rank
                FROM Employee e
                JOIN Department d ON e.departmentId = d.id
            )
            WHERE salary_rank <= p_n
            ORDER BY Department, salary_rank;
    END get_top_n_salaries;

    FUNCTION get_dept_top_salary(p_dept_id IN NUMBER) RETURN NUMBER IS
        v_max_salary NUMBER;
    BEGIN
        SELECT MAX(salary) INTO v_max_salary
        FROM Employee
        WHERE departmentId = p_dept_id;

        RETURN v_max_salary;
    EXCEPTION
        WHEN NO_DATA_FOUND THEN
            RETURN NULL;
    END get_dept_top_salary;

END dept_salary_pkg;
/

-- ============================================================================
-- 196. Delete Duplicate Emails - Multiple Approaches
-- ============================================================================

-- Procedure 1: Using ROWID (fastest)
CREATE OR REPLACE PROCEDURE delete_duplicate_emails_v1 IS
    v_deleted_count NUMBER := 0;
BEGIN
    DELETE FROM Person
    WHERE ROWID NOT IN (
        SELECT MIN(ROWID)
        FROM Person
        GROUP BY email
    );

    v_deleted_count := SQL%ROWCOUNT;
    COMMIT;

    leetcode_utils.log_message('Deleted ' || v_deleted_count || ' duplicate emails (v1)');
EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        leetcode_utils.log_message('Error in delete_duplicate_emails_v1: ' || SQLERRM);
        RAISE;
END delete_duplicate_emails_v1;
/

-- Procedure 2: Using cursor and tracking
CREATE OR REPLACE PROCEDURE delete_duplicate_emails_v2 IS
    CURSOR c_duplicates IS
        SELECT email, MIN(id) as min_id, COUNT(*) as dup_count
        FROM Person
        GROUP BY email
        HAVING COUNT(*) > 1;

    v_deleted_count NUMBER := 0;
    v_total_deleted NUMBER := 0;
BEGIN
    FOR rec IN c_duplicates LOOP
        DELETE FROM Person
        WHERE email = rec.email
          AND id > rec.min_id;

        v_deleted_count := SQL%ROWCOUNT;
        v_total_deleted := v_total_deleted + v_deleted_count;

        leetcode_utils.log_message(
            'Email: ' || rec.email ||
            ', Kept ID: ' || rec.min_id ||
            ', Deleted: ' || v_deleted_count
        );
    END LOOP;

    COMMIT;
    leetcode_utils.log_message('Total deleted: ' || v_total_deleted);
EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        leetcode_utils.log_message('Error: ' || SQLERRM);
        RAISE;
END delete_duplicate_emails_v2;
/

-- Function to preview duplicates before deletion
CREATE OR REPLACE FUNCTION preview_duplicate_emails
RETURN SYS_REFCURSOR
IS
    v_cursor SYS_REFCURSOR;
BEGIN
    OPEN v_cursor FOR
        SELECT email, COUNT(*) as duplicate_count, MIN(id) as keep_id
        FROM Person
        GROUP BY email
        HAVING COUNT(*) > 1
        ORDER BY duplicate_count DESC, email;

    RETURN v_cursor;
END preview_duplicate_emails;
/

-- ============================================================================
-- 601. Human Traffic of Stadium - Advanced
-- ============================================================================

CREATE OR REPLACE PROCEDURE find_high_traffic_periods(
    p_min_people IN NUMBER DEFAULT 100,
    p_min_consecutive IN NUMBER DEFAULT 3,
    p_cursor OUT SYS_REFCURSOR
) IS
BEGIN
    OPEN p_cursor FOR
        WITH high_traffic AS (
            SELECT id, visit_date, people,
                   id - ROW_NUMBER() OVER (ORDER BY id) as grp
            FROM Stadium
            WHERE people >= p_min_people
        ),
        consecutive_groups AS (
            SELECT grp,
                   COUNT(*) as consecutive_days,
                   MIN(id) as start_id,
                   MAX(id) as end_id
            FROM high_traffic
            GROUP

In [ ]:
-- ============================================================================
-- COMPREHENSIVE PL/SQL SOLUTIONS FOR LEETCODE SQL PROBLEMS
-- ============================================================================

-- ============================================================================
-- UTILITY FUNCTIONS AND TYPES
-- ============================================================================

-- Create a package for common utilities
CREATE OR REPLACE PACKAGE leetcode_utils AS
    TYPE num_array IS TABLE OF NUMBER;
    TYPE varchar_array IS TABLE OF VARCHAR2(1000);
    TYPE date_array IS TABLE OF DATE;

    PROCEDURE log_message(p_message VARCHAR2);
    FUNCTION get_nth_value(p_array num_array, p_position NUMBER) RETURN NUMBER;
END leetcode_utils;
/

CREATE OR REPLACE PACKAGE BODY leetcode_utils AS
    PROCEDURE log_message(p_message VARCHAR2) IS
    BEGIN
        DBMS_OUTPUT.PUT_LINE(TO_CHAR(SYSTIMESTAMP, 'YYYY-MM-DD HH24:MI:SS') || ' - ' || p_message);
    END log_message;

    FUNCTION get_nth_value(p_array num_array, p_position NUMBER) RETURN NUMBER IS
    BEGIN
        IF p_position <= 0 OR p_position > p_array.COUNT THEN
            RETURN NULL;
        END IF;
        RETURN p_array(p_position);
    END get_nth_value;
END leetcode_utils;
/

-- ============================================================================
-- 177. Nth Highest Salary - Multiple Implementations
-- ============================================================================

-- Function version 1: Simple approach
CREATE OR REPLACE FUNCTION get_nth_highest_salary_v1(p_n IN NUMBER)
RETURN NUMBER
DETERMINISTIC
IS
    v_salary NUMBER;
BEGIN
    SELECT salary INTO v_salary
    FROM (
        SELECT DISTINCT salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) as rank
        FROM Employee
    )
    WHERE rank = p_n;

    RETURN v_salary;
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RETURN NULL;
    WHEN OTHERS THEN
        leetcode_utils.log_message('Error in get_nth_highest_salary_v1: ' || SQLERRM);
        RETURN NULL;
END get_nth_highest_salary_v1;
/

-- Function version 2: With validation
CREATE OR REPLACE FUNCTION get_nth_highest_salary_v2(p_n IN NUMBER)
RETURN NUMBER
IS
    v_salary NUMBER;
    v_max_rank NUMBER;
BEGIN
    -- Validate input
    IF p_n IS NULL OR p_n <= 0 THEN
        RETURN NULL;
    END IF;

    -- Check if N exists
    SELECT MAX(rank) INTO v_max_rank
    FROM (
        SELECT DISTINCT salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) as rank
        FROM Employee
    );

    IF p_n > v_max_rank THEN
        RETURN NULL;
    END IF;

    -- Get the salary
    SELECT salary INTO v_salary
    FROM (
        SELECT DISTINCT salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) as rank
        FROM Employee
    )
    WHERE rank = p_n;

    RETURN v_salary;
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RETURN NULL;
    WHEN OTHERS THEN
        RAISE;
END get_nth_highest_salary_v2;
/

-- Procedure version with OUT parameter
CREATE OR REPLACE PROCEDURE get_nth_salary_proc(
    p_n IN NUMBER,
    p_salary OUT NUMBER,
    p_status OUT VARCHAR2
) IS
BEGIN
    p_status := 'SUCCESS';

    SELECT salary INTO p_salary
    FROM (
        SELECT DISTINCT salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) as rn
        FROM Employee
    )
    WHERE rn = p_n;

EXCEPTION
    WHEN NO_DATA_FOUND THEN
        p_salary := NULL;
        p_status := 'NOT_FOUND';
    WHEN OTHERS THEN
        p_salary := NULL;
        p_status := 'ERROR: ' || SQLERRM;
END get_nth_salary_proc;
/

-- ============================================================================
-- 178. Rank Scores - Procedure with Output
-- ============================================================================

CREATE OR REPLACE PROCEDURE rank_all_scores(
    p_cursor OUT SYS_REFCURSOR
) IS
BEGIN
    OPEN p_cursor FOR
        SELECT score,
               DENSE_RANK() OVER (ORDER BY score DESC) as rank
        FROM Scores
        ORDER BY score DESC;
END rank_all_scores;
/

-- Procedure to materialize ranked scores
CREATE OR REPLACE PROCEDURE materialize_ranked_scores IS
    v_count NUMBER := 0;
BEGIN
    -- Create or replace temp table
    EXECUTE IMMEDIATE 'TRUNCATE TABLE Scores_Ranked';

    INSERT INTO Scores_Ranked (score_id, score, rank)
    SELECT id, score,
           DENSE_RANK() OVER (ORDER BY score DESC) as rank
    FROM Scores;

    v_count := SQL%ROWCOUNT;
    COMMIT;

    leetcode_utils.log_message('Ranked ' || v_count || ' scores');
EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        leetcode_utils.log_message('Error: ' || SQLERRM);
        RAISE;
END materialize_ranked_scores;
/

-- ============================================================================
-- 180. Consecutive Numbers - Advanced Implementation
-- ============================================================================

CREATE OR REPLACE FUNCTION find_consecutive_numbers(p_count IN NUMBER DEFAULT 3)
RETURN SYS_REFCURSOR
IS
    v_cursor SYS_REFCURSOR;
BEGIN
    OPEN v_cursor FOR
        WITH numbered_logs AS (
            SELECT num,
                   LAG(num, 1) OVER (ORDER BY id) as prev1,
                   LAG(num, 2) OVER (ORDER BY id) as prev2,
                   LEAD(num, 1) OVER (ORDER BY id) as next1,
                   LEAD(num, 2) OVER (ORDER BY id) as next2
            FROM Logs
        )
        SELECT DISTINCT num as ConsecutiveNums
        FROM numbered_logs
        WHERE (num = prev1 AND num = prev2) OR
              (num = prev1 AND num = next1) OR
              (num = next1 AND num = next2);

    RETURN v_cursor;
END find_consecutive_numbers;
/

-- Procedure with collection output
CREATE OR REPLACE PROCEDURE get_consecutive_numbers(
    p_min_consecutive IN NUMBER DEFAULT 3,
    p_numbers OUT leetcode_utils.num_array
) IS
    CURSOR c_logs IS
        SELECT id, num FROM Logs ORDER BY id;

    TYPE t_log_rec IS RECORD (
        id NUMBER,
        num NUMBER
    );
    TYPE t_log_table IS TABLE OF t_log_rec;

    v_logs t_log_table;
    v_consecutive_count NUMBER;
    v_current_num NUMBER;
    v_result_set leetcode_utils.num_array := leetcode_utils.num_array();
BEGIN
    -- Load all logs
    SELECT id, num
    BULK COLLECT INTO v_logs
    FROM Logs
    ORDER BY id;

    -- Find consecutive numbers
    FOR i IN 1..v_logs.COUNT - (p_min_consecutive - 1) LOOP
        v_current_num := v_logs(i).num;
        v_consecutive_count := 1;

        FOR j IN i+1..LEAST(i + p_min_consecutive - 1, v_logs.COUNT) LOOP
            IF v_logs(j).num = v_current_num THEN
                v_consecutive_count := v_consecutive_count + 1;
            ELSE
                EXIT;
            END IF;
        END LOOP;

        IF v_consecutive_count >= p_min_consecutive THEN
            -- Check if already in result
            IF NOT v_result_set.EXISTS(1) OR v_current_num NOT MEMBER OF v_result_set THEN
                v_result_set.EXTEND;
                v_result_set(v_result_set.LAST) := v_current_num;
            END IF;
        END IF;
    END LOOP;

    p_numbers := v_result_set;
END get_consecutive_numbers;
/

-- ============================================================================
-- 184 & 185. Department Salaries - Package
-- ============================================================================

CREATE OR REPLACE PACKAGE dept_salary_pkg AS
    PROCEDURE get_highest_salaries(p_cursor OUT SYS_REFCURSOR);
    PROCEDURE get_top_n_salaries(p_n IN NUMBER, p_cursor OUT SYS_REFCURSOR);
    FUNCTION get_dept_top_salary(p_dept_id IN NUMBER) RETURN NUMBER;
END dept_salary_pkg;
/

CREATE OR REPLACE PACKAGE BODY dept_salary_pkg AS

    PROCEDURE get_highest_salaries(p_cursor OUT SYS_REFCURSOR) IS
    BEGIN
        OPEN p_cursor FOR
            SELECT d.name as Department,
                   e.name as Employee,
                   e.salary as Salary
            FROM Employee e
            JOIN Department d ON e.departmentId = d.id
            WHERE (e.departmentId, e.salary) IN (
                SELECT departmentId, MAX(salary)
                FROM Employee
                GROUP BY departmentId
            )
            ORDER BY d.name, e.name;
    END get_highest_salaries;

    PROCEDURE get_top_n_salaries(p_n IN NUMBER, p_cursor OUT SYS_REFCURSOR) IS
    BEGIN
        OPEN p_cursor FOR
            SELECT Department, Employee, Salary
            FROM (
                SELECT d.name as Department,
                       e.name as Employee,
                       e.salary as Salary,
                       DENSE_RANK() OVER (
                           PARTITION BY d.id
                           ORDER BY e.salary DESC
                       ) as salary_rank
                FROM Employee e
                JOIN Department d ON e.departmentId = d.id
            )
            WHERE salary_rank <= p_n
            ORDER BY Department, salary_rank;
    END get_top_n_salaries;

    FUNCTION get_dept_top_salary(p_dept_id IN NUMBER) RETURN NUMBER IS
        v_max_salary NUMBER;
    BEGIN
        SELECT MAX(salary) INTO v_max_salary
        FROM Employee
        WHERE departmentId = p_dept_id;

        RETURN v_max_salary;
    EXCEPTION
        WHEN NO_DATA_FOUND THEN
            RETURN NULL;
    END get_dept_top_salary;

END dept_salary_pkg;
/

-- ============================================================================
-- 196. Delete Duplicate Emails - Multiple Approaches
-- ============================================================================

-- Procedure 1: Using ROWID (fastest)
CREATE OR REPLACE PROCEDURE delete_duplicate_emails_v1 IS
    v_deleted_count NUMBER := 0;
BEGIN
    DELETE FROM Person
    WHERE ROWID NOT IN (
        SELECT MIN(ROWID)
        FROM Person
        GROUP BY email
    );

    v_deleted_count := SQL%ROWCOUNT;
    COMMIT;

    leetcode_utils.log_message('Deleted ' || v_deleted_count || ' duplicate emails (v1)');
EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        leetcode_utils.log_message('Error in delete_duplicate_emails_v1: ' || SQLERRM);
        RAISE;
END delete_duplicate_emails_v1;
/

-- Procedure 2: Using cursor and tracking
CREATE OR REPLACE PROCEDURE delete_duplicate_emails_v2 IS
    CURSOR c_duplicates IS
        SELECT email, MIN(id) as min_id, COUNT(*) as dup_count
        FROM Person
        GROUP BY email
        HAVING COUNT(*) > 1;

    v_deleted_count NUMBER := 0;
    v_total_deleted NUMBER := 0;
BEGIN
    FOR rec IN c_duplicates LOOP
        DELETE FROM Person
        WHERE email = rec.email
          AND id > rec.min_id;

        v_deleted_count := SQL%ROWCOUNT;
        v_total_deleted := v_total_deleted + v_deleted_count;

        leetcode_utils.log_message(
            'Email: ' || rec.email ||
            ', Kept ID: ' || rec.min_id ||
            ', Deleted: ' || v_deleted_count
        );
    END LOOP;

    COMMIT;
    leetcode_utils.log_message('Total deleted: ' || v_total_deleted);
EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        leetcode_utils.log_message('Error: ' || SQLERRM);
        RAISE;
END delete_duplicate_emails_v2;
/

-- Function to preview duplicates before deletion
CREATE OR REPLACE FUNCTION preview_duplicate_emails
RETURN SYS_REFCURSOR
IS
    v_cursor SYS_REFCURSOR;
BEGIN
    OPEN v_cursor FOR
        SELECT email, COUNT(*) as duplicate_count, MIN(id) as keep_id
        FROM Person
        GROUP BY email
        HAVING COUNT(*) > 1
        ORDER BY duplicate_count DESC, email;

    RETURN v_cursor;
END preview_duplicate_emails;
/

-- ============================================================================
-- 601. Human Traffic of Stadium - Advanced
-- ============================================================================

CREATE OR REPLACE PROCEDURE find_high_traffic_periods(
    p_min_people IN NUMBER DEFAULT 100,
    p_min_consecutive IN NUMBER DEFAULT 3,
    p_cursor OUT SYS_REFCURSOR
) IS
BEGIN
    OPEN p_cursor FOR
        WITH high_traffic AS (
            SELECT id, visit_date, people,
                   id - ROW_NUMBER() OVER (ORDER BY id) as grp
            FROM Stadium
            WHERE people >= p_min_people
        ),
        consecutive_groups AS (
            SELECT grp,
                   COUNT(*) as consecutive_days,
                   MIN(id) as start_id,
                   MAX(id) as end_id
            FROM high_traffic
            GROUP BY grp
            HAVING COUNT(*) >= p_min_consecutive
        )
        SELECT s.id, s.visit_date, s.people
        FROM Stadium s
        JOIN consecutive_groups cg
            ON s.id BETWEEN cg.start_id AND cg.end_id
        ORDER BY s.visit_date;
END find_high_traffic_periods;
/

-- ============================================================================
-- 626. Exchange Seats - Procedure
-- ============================================================================

CREATE OR REPLACE PROCEDURE exchange_student_seats IS
    TYPE t_seat_rec IS RECORD (
        id NUMBER,
        student VARCHAR2(100)
    );
    TYPE t_seats_table IS TABLE OF t_seat_rec INDEX BY PLS_INTEGER;

    v_seats t_seats_table;
    v_idx PLS_INTEGER := 1;
    v_temp_student VARCHAR2(100);
BEGIN
    -- Load all seats
    FOR rec IN (SELECT id, student FROM Seat ORDER BY id) LOOP
        v_seats(v_idx).id := rec.id;
        v_seats(v_idx).student := rec.student;
        v_idx := v_idx + 1;
    END LOOP;

    -- Swap in pairs
    FOR i IN 1..v_seats.COUNT LOOP
        IF MOD(i, 2) = 1 AND i < v_seats.COUNT THEN
            v_temp_student := v_seats(i).student;
            v_seats(i).student := v_seats(i + 1).student;
            v_seats(i + 1).student := v_temp_student;
        END IF;
    END LOOP;

    -- Update the table
    FOR i IN 1..v_seats.COUNT LOOP
        UPDATE Seat
        SET student = v_seats(i).student
        WHERE id = v_seats(i).id;
    END LOOP;

    COMMIT;
    leetcode_utils.log_message('Exchanged ' || v_seats.COUNT || ' seats');
EXCEPTION
    WHEN OTHERS THEN
        ROLLBACK;
        RAISE;
END exchange_student_seats;
/

-- Function to preview exchange without modifying
CREATE OR REPLACE FUNCTION preview_seat_exchange
RETURN SYS_REFCURSOR
IS
    v_cursor SYS_REFCURSOR;
BEGIN
    OPEN v_cursor FOR
        SELECT
            CASE
                WHEN MOD(id, 2) = 1 AND id != (SELECT MAX(id) FROM Seat)
                    THEN id + 1
                WHEN MOD(id, 2) = 0
                    THEN id - 1
                ELSE id
            END as new_id,
            student
        FROM Seat
        ORDER BY new_id;

    RETURN v_cursor;
END preview_seat_exchange;
/

-- ============================================================================
-- 1341. Movie Rating - Comprehensive Solution
-- ============================================================================

CREATE OR REPLACE PROCEDURE get_movie_rating_insights(
    p_most_ratings_user OUT VARCHAR2,
    p_highest_rated_movie OUT VARCHAR2,
    p_target_month IN VARCHAR2 DEFAULT '2020-02'
) IS
    v_user_name VARCHAR2(100);
    v_movie_title VARCHAR2(200);
BEGIN
    -- Get user with most ratings
    SELECT name INTO v_user_name
    FROM (
        SELECT u.name, COUNT(*) as rating_count
        FROM MovieRating mr
        JOIN Users u ON mr.user_id = u.user_id
        GROUP BY u.name
        ORDER BY rating_count DESC, u.name
    )
    WHERE ROWNUM = 1;

    p_most_ratings_user := v_user_name;

    -- Get highest rated movie in target month
    SELECT title INTO v_movie_title
    FROM (
        SELECT m.title, AVG(mr.rating) as avg_rating
        FROM MovieRating mr
        JOIN Movies m ON mr.movie_id = m.movie_id
        WHERE TO_CHAR(mr.created_at, 'YYYY-MM') = p_target_month
        GROUP BY m.title
        ORDER BY avg_rating DESC, m.title
    )
    WHERE ROWNUM = 1;

    p_highest_rated_movie := v_movie_title;

EXCEPTION
    WHEN NO_DATA_FOUND THEN
        p_most_ratings_user := NULL;
        p_highest_rated_movie := NULL;
    WHEN OTHERS THEN
        RAISE;
END get_movie_rating_insights;
/

-- ============================================================================
-- 1393. Capital Gain/Loss - Package
-- ============================================================================

CREATE OR REPLACE PACKAGE stock_analysis_pkg AS
    TYPE t_stock_result IS RECORD (
        stock_name VARCHAR2(100),
        capital_gain_loss NUMBER
    );
    TYPE t_stock_results IS TABLE OF t_stock_result;

    FUNCTION calculate_gains RETURN t_stock_results PIPELINED;
    PROCEDURE calculate_and_store;
    FUNCTION get_stock_gain(p_stock_name VARCHAR2) RETURN NUMBER;
END stock_analysis_pkg;
/

CREATE OR REPLACE PACKAGE BODY stock_analysis_pkg AS

    FUNCTION calculate_gains RETURN t_stock_results PIPELINED IS
        v_result t_stock_result;
    BEGIN
        FOR rec IN (
            SELECT stock_name,
                   SUM(CASE WHEN operation = 'Buy' THEN -price ELSE price END) as gain_loss
            FROM Stocks
            GROUP BY stock_name
        ) LOOP
            v_result.stock_name := rec.stock_name;
            v_result.capital_gain_loss := rec.gain_loss;
            PIPE ROW(v_result);
        END LOOP;

        RETURN;
    END calculate_gains;

    PROCEDURE calculate_and_store IS
    BEGIN
        MERGE INTO Stock_Gains sg
        USING (
            SELECT stock_name,
                   SUM(CASE WHEN operation = 'Buy' THEN -price ELSE price END) as gain_loss
            FROM Stocks
            GROUP BY stock_name
        ) s
        ON (sg.stock_name = s.stock_name)
        WHEN MATCHED THEN
            UPDATE SET sg.gain_loss = s.gain_loss, sg.last_updated = SYSDATE
        WHEN NOT MATCHED THEN
            INSERT (stock_name, gain_loss, last_updated)
            VALUES (s.stock_name, s.gain_loss, SYSDATE);

        COMMIT;
    END calculate_and_store;

    FUNCTION get_stock_gain(p_stock_name VARCHAR2) RETURN NUMBER IS
        v_gain NUMBER;
    BEGIN
        SELECT SUM(CASE WHEN operation = 'Buy' THEN -price ELSE price END)
        INTO v_gain
        FROM Stocks
        WHERE stock_name = p_stock_name;

        RETURN v_gain;
    EXCEPTION
        WHEN NO_DATA_FOUND THEN
            RETURN 0;
    END get_stock_gain;

END stock_analysis_pkg;
/

-- ============================================================================
-- 1454. Active Users - Advanced Implementation
-- ============================================================================

CREATE OR REPLACE PROCEDURE find_active_users(
    p_consecutive_days IN NUMBER DEFAULT 5,
    p_cursor OUT SYS_REFCURSOR
) IS
BEGIN
    OPEN p_cursor FOR
        WITH user_dates AS (
            SELECT DISTINCT user_id, activity_date
            FROM Logins
        ),
        numbered AS (
            SELECT user_id,
                   activity_date,
                   activity_date - ROW_NUMBER() OVER (
                       PARTITION BY user_id
                       ORDER BY activity_date
                   ) as date_group
            FROM user_dates
        ),
        consecutive_logins AS (
            SELECT user_id,
                   date_group,
                   COUNT(*) as consecutive_count
            FROM numbered
            GROUP BY user_id, date_group
            HAVING COUNT(*) >= p_consecutive_days
        )
        SELECT DISTINCT u.id, u.name
        FROM Users u
        JOIN consecutive_logins cl ON u.id = cl.user_id
        ORDER BY u.id;
END find_active_users;
/

-- Function to check if specific user is active
CREATE OR REPLACE FUNCTION is_user_active(
    p_user_id IN NUMBER,
    p_consecutive_days IN NUMBER DEFAULT 5
) RETURN VARCHAR2
IS
    v_is_active NUMBER;
BEGIN
    SELECT COUNT(*)
    INTO v_is_active
    FROM (
        WITH user_dates AS (
            SELECT DISTINCT activity_date
            FROM Logins
            WHERE user_id = p_user_id
        ),
        numbered AS (
            SELECT activity_date,
                   activity_date - ROW_NUMBER() OVER (ORDER BY activity_date) as grp
            FROM user_dates
        )
        SELECT 1
        FROM numbered
        GROUP BY grp
        HAVING COUNT(*) >= p_consecutive_days
    )
    WHERE ROWNUM = 1;

    RETURN CASE WHEN v_is_active > 0 THEN 'YES' ELSE 'NO' END;
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RETURN 'NO';
END is_user_active;
/

-- ============================================================================
-- HARD PROBLEMS - Advanced PL/SQL
-- ============================================================================

-- 603. Consecutive Available Seats
CREATE OR REPLACE FUNCTION find_consecutive_seats(
    p_min_consecutive IN NUMBER DEFAULT 2
) RETURN SYS_REFCURSOR
IS
    v_cursor SYS_REFCURSOR;
BEGIN
    OPEN v_cursor FOR
        SELECT DISTINCT c1.seat_id
        FROM Cinema c1
        JOIN Cinema c2
            ON ABS(c1.seat_id - c2.seat_id) = 1
            AND c1.free = 1
            AND c2.free = 1
        ORDER BY c1.seat_id;

    RETURN v_cursor;
END find_consecutive_seats;
/

-- 1336. Number of Transactions per Visit
CREATE OR REPLACE PROCEDURE analyze_transactions_per_visit(
    p_cursor OUT SYS_REFCURSOR
) IS
BEGIN
    OPEN p_cursor FOR
        WITH visit_trans_count AS (
            SELECT v.user_id, v.visit_date,
                   COUNT(t.transaction_date) as trans_count
            FROM Visits v
            LEFT JOIN Transactions t
                ON v.user_id = t.user_id
                AND v.visit_date = t.transaction_date
            GROUP BY v.user_id, v.visit_date
        ),
        numbers AS (
            SELECT LEVEL - 1 as n
            FROM DUAL
            CONNECT BY LEVEL <= (
                SELECT NVL(MAX(trans_count), 0) + 1
                FROM visit_trans_count
            )
        )
        SELECT n.n as transactions_count,
               NVL(COUNT(vtc.trans_count), 0) as visits_count
        FROM numbers n
        LEFT JOIN visit_trans_count vtc ON n.n = vtc.trans_count
        GROUP BY n.n
        ORDER BY n.n;
END analyze_transactions_per_visit;
/

-- 1369. Get Second Most Recent Activity
CREATE OR REPLACE PROCEDURE get_second_recent_activities(
    p_cursor OUT SYS_REFCURSOR
) IS
BEGIN
    OPEN p_cursor FOR
        WITH ranked_activities AS (
            SELECT username, activity, startDate, endDate,
                   ROW_NUMBER() OVER (
                       PARTITION BY username
                       ORDER BY startDate DESC
                   ) as rn,
                   COUNT(*) OVER (PARTITION BY username) as total_count
            FROM UserActivity
        )
        SELECT username, activity, startDate, endDate
        FROM ranked_activities
        WHERE rn = 2 OR (total_count = 1 AND rn = 1)
        ORDER BY username;
END get_second_recent_activities;
/

-- 1412. Find Quiet Students
CREATE OR REPLACE PROCEDURE find_quiet_students(
    p_cursor OUT SYS_REFCURSOR
) IS
BEGIN
    OPEN p_cursor FOR
        WITH exam_scores AS (
            SELECT exam_id,
                   MAX(score) as max_score,
                   MIN(score) as min_score
            FROM Exam
            GROUP BY exam_id
        ),
        noisy_students AS (
            SELECT DISTINCT e.student_id
            FROM Exam e
            JOIN exam_scores es ON e.exam_id = es.exam_id
            WHERE e.score = es.max_score OR e.score = es.min_score
        )
        SELECT DISTINCT s.student_id, s.student_name
        FROM Student s
        JOIN Exam e ON s.student_id = e.student_id
        WHERE s.student_id NOT IN (SELECT student_id FROM noisy_students)
        ORDER BY s.student_id;
END find_quiet_students;
/

-- ============================================================================
-- COMPREHENSIVE ANALYTICS PACKAGE
-- ============================================================================

CREATE OR REPLACE PACKAGE analytics_pkg AS
    -- Performance metrics
    PROCEDURE analyze_query_performance(p_query_name VARCHAR2);

    -- Department analytics
    PROCEDURE get_dept_statistics(
        p_dept_id IN NUMBER,
        p_avg_salary OUT NUMBER,
        p_employee_count OUT NUMBER,
        p_max_salary OUT NUMBER
    );

    -- User activity analytics
    FUNCTION get_user_engagement_score(p_user_id NUMBER) RETURN NUMBER;

    -- Transaction analytics
    PROCEDURE monthly_transaction_report(p_month VARCHAR2, p_cursor OUT SYS_REFCURSOR);

END analytics_pkg;
/

CREATE OR REPLACE PACKAGE BODY analytics_pkg AS

    PROCEDURE analyze_query_performance(p_query_name VARCHAR2) IS
        v_avg_quality NUMBER;
        v_poor_percentage NUMBER;
        v_total_queries NUMBER;
    BEGIN
        SELECT
            ROUND(AVG(rating * 1.0 / position), 2),
            ROUND(SUM(CASE WHEN rating < 3 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2),
            COUNT(*)
        INTO v_avg_quality, v_poor_percentage, v_total_queries
        FROM Queries
        WHERE query_name = p_query_name;

        DBMS_OUTPUT.PUT_LINE('Query: ' || p_query_name);
        DBMS_OUTPUT.PUT_LINE('Quality Score: ' || v_avg_quality);
        DBMS_OUTPUT.PUT_LINE('Poor Query %: ' || v_poor_percentage);
        DBMS_OUTPUT.PUT_LINE('Total Queries: ' || v_total_queries);
    EXCEPTION
        WHEN NO_DATA_FOUND THEN
            DBMS_OUTPUT.PUT_LINE('No data found for query: ' || p_query_name);
    END analyze_query_performance;

    PROCEDURE get_dept_statistics(
        p_dept_id IN NUMBER,
        p_avg_salary OUT NUMBER,
        p_employee_count OUT NUMBER,
        p_max_salary OUT NUMBER
    ) IS
    BEGIN
        SELECT
            AVG(salary),
            COUNT(*),
            MAX(salary)
        INTO p_avg_salary, p_employee_count, p_max_salary
        FROM Employee
        WHERE departmentId = p_dept_id;
    EXCEPTION
        WHEN NO_DATA_FOUND THEN
            p_avg_salary := 0;
            p_employee_count := 0;
            p_max_salary := 0;
    END get_dept_statistics;

    FUNCTION get_user_engagement_score(p_user_id NUMBER) RETURN NUMBER IS
        v_activity_count NUMBER;
        v_days_active NUMBER;
        v_score NUMBER;
    BEGIN
        SELECT COUNT(*), COUNT(DISTINCT activity_date)
        INTO v_activity_count, v_days_active
        FROM Activity
        WHERE user_id = p_user_id;

        v_score := (v_activity_count * 0.6) + (v_days_active * 0.4);
        RETURN ROUND(v_score, 2);
    EXCEPTION
        WHEN NO_DATA_FOUND THEN
            RETURN 0;
    END get_user_engagement_score;

    PROCEDURE monthly_transaction_report(
        p_month VARCHAR2,
        p_cursor OUT SYS_REFCURSOR
    ) IS
    BEGIN
        OPEN p_cursor FOR
            SELECT
                country,
                COUNT(*) as total_transactions,
                SUM(CASE WHEN state = 'approved' THEN 1 ELSE 0 END) as approved,
                SUM(amount) as total_amount,
                SUM(CASE WHEN state = 'approved' THEN amount ELSE 0 END) as approved_amount,
                ROUND(AVG(amount), 2) as avg_amount
            FROM Transactions
            WHERE TO_CHAR(trans_date, 'YYYY-MM') = p_month
            GROUP BY country
            ORDER BY total_transactions DESC;
    END monthly_transaction_report;

END analytics_pkg;
/

-- ============================================================================
-- TESTING AND UTILITY PROCEDURES
-- ============================================================================

CREATE OR REPLACE PROCEDURE test_all_solutions IS
    v_salary NUMBER;
    v_status VARCHAR2(100);
    v_cursor SYS_REFCURSOR;
    v_numbers leetcode_utils.num_array;
BEGIN
    DBMS_OUTPUT.PUT_LINE('=== Testing LeetCode Solutions ===');
    DBMS_OUTPUT.PUT_LINE('');

    -- Test Nth highest salary
    DBMS_OUTPUT.PUT_LINE('1. Testing Nth Highest Salary:');
    v_salary := get_nth_highest_salary_v1(2);
    DBMS_OUTPUT.PUT_LINE('2nd highest salary: ' || NVL(TO_CHAR(v_salary), 'NULL'));
    DBMS_OUTPUT.PUT_LINE('');

    -- Test consecutive numbers
    DBMS_OUTPUT.PUT_LINE('2. Testing Consecutive Numbers:');
    get_consecutive_numbers(3, v_numbers);
    IF v_numbers IS NOT NULL AND v_numbers.COUNT > 0 THEN
        FOR i IN 1..v_numbers.COUNT LOOP
            DBMS_OUTPUT.PUT_LINE('Consecutive number: ' || v_numbers(i));
        END LOOP;
    END IF;
    DBMS_OUTPUT.PUT_LINE('');

    DBMS_OUTPUT.PUT_LINE('=== All Tests Complete ===');
END test_all_solutions;
/

In [ ]:
# Complete LeetCode SQL Solutions Reference Guide

## Overview
This comprehensive guide provides solutions for LeetCode SQL problems using **Oracle SQL**, **PL/SQL**, and **Python (Pandas)**.

---

## 📋 Table of Contents

### Medium Difficulty Problems
1. [177. Nth Highest Salary](#177-nth-highest-salary)
2. [178. Rank Scores](#178-rank-scores)
3. [180. Consecutive Numbers](#180-consecutive-numbers)
4. [184. Department Highest Salary](#184-department-highest-salary)
5. [185. Department Top Three Salaries](#185-department-top-three-salaries)
6. [196. Delete Duplicate Emails](#196-delete-duplicate-emails)
7. [601. Human Traffic of Stadium](#601-human-traffic-of-stadium)
8. [626. Exchange Seats](#626-exchange-seats)
9. [1341. Movie Rating](#1341-movie-rating)
10. [1393. Capital Gain/Loss](#1393-capital-gainloss)
11. [1454. Active Users](#1454-active-users)

### Hard Difficulty Problems
12. [603. Consecutive Available Seats](#603-consecutive-available-seats)
13. [1336. Number of Transactions per Visit](#1336-number-of-transactions-per-visit)
14. [1369. Get Second Most Recent Activity](#1369-get-second-most-recent-activity)
15. [1412. Find Quiet Students](#1412-find-quiet-students)

### Easy Problems (Quick Reference)
16. Multiple easy problems covered

---

## 🔑 Key Concepts by Problem Type

### Window Functions
- **RANK() / DENSE_RANK() / ROW_NUMBER()**: Used in problems 177, 178, 184, 185, 1369
- **LAG() / LEAD()**: Used in problems 180, 197, 626
- **Partition By**: Essential for department-wise analysis (184, 185)

### Date/Time Operations
- **Date arithmetic**: Problems 197, 550, 1454
- **Date grouping**: Problems 1193, 1174
- **Consecutive date patterns**: Problems 601, 1454

### Aggregation Patterns
- **GROUP BY with HAVING**: Problems 570, 596, 619
- **Conditional aggregation**: Problems 1211, 1193, 1393
- **Multiple aggregations**: Most reporting problems

### Self-Joins
- **Consecutive patterns**: Problems 180, 603
- **Hierarchical data**: Problems 181, 570

---

## 💡 Solution Strategies

### 1. **Nth Highest Salary Pattern**
```sql
-- Strategy: Use DENSE_RANK or OFFSET
SELECT DISTINCT salary
FROM (
    SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) as rnk
    FROM Employee
)
WHERE rnk = N;
```

**When to use:**
- Finding top/bottom N values
- Handling ties with DENSE_RANK
- Skipping duplicates

---

### 2. **Consecutive Pattern Detection**
```sql
-- Strategy: Use LAG/LEAD or row number grouping
SELECT DISTINCT num
FROM (
    SELECT num,
           LAG(num, 1) OVER (ORDER BY id) as prev,
           LAG(num, 2) OVER (ORDER BY id) as prev2
    FROM Logs
)
WHERE num = prev AND num = prev2;
```

**When to use:**
- Finding consecutive sequences
- Detecting patterns in ordered data
- Grouping related events

---

### 3. **Remove Duplicates Pattern**
```sql
-- Strategy: Keep MIN(id), delete rest
DELETE FROM Person
WHERE ROWID NOT IN (
    SELECT MIN(ROWID)
    FROM Person
    GROUP BY email
);
```

**When to use:**
- Data deduplication
- Keeping earliest/latest record
- Cleaning data issues

---

### 4. **Department/Group Top N Pattern**
```sql
-- Strategy: DENSE_RANK with PARTITION BY
SELECT Department, Employee, Salary
FROM (
    SELECT d.name as Department,
           e.name as Employee,
           e.salary as Salary,
           DENSE_RANK() OVER (PARTITION BY d.id ORDER BY e.salary DESC) as rnk
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
)
WHERE rnk <= 3;
```

**When to use:**
- Top performers per category
- Regional rankings
- Category-wise analysis

---

## 🎯 Common Pitfalls and Solutions

### Pitfall 1: Handling NULLs
**Problem:** NULLs in joins or comparisons
```sql
-- ❌ Wrong
WHERE referee_id != 2

-- ✅ Correct
WHERE referee_id IS NULL OR referee_id != 2
```

### Pitfall 2: Date Comparisons
**Problem:** Including time component
```sql
-- ❌ Wrong
WHERE date_column = '2020-01-01'

-- ✅ Correct
WHERE TRUNC(date_column) = DATE '2020-01-01'
-- OR
WHERE date_column >= DATE '2020-01-01'
  AND date_column < DATE '2020-01-02'
```

### Pitfall 3: Division by Zero
**Problem:** Aggregate functions with empty groups
```sql
-- ❌ Wrong
SELECT AVG(score)

-- ✅ Correct
SELECT COALESCE(AVG(score), 0)
-- OR
SELECT AVG(score) / NULLIF(COUNT(*), 0)
```

### Pitfall 4: Duplicate Handling
**Problem:** Not accounting for ties
```sql
-- ❌ Wrong (RANK leaves gaps)
RANK() OVER (ORDER BY score DESC)

-- ✅ Correct (DENSE_RANK no gaps)
DENSE_RANK() OVER (ORDER BY score DESC)
```

---

## 🔧 Oracle SQL Specific Tips

### 1. **ROWID for Fastest Deletes**
```sql
-- Most efficient delete in Oracle
DELETE FROM table_name
WHERE ROWID NOT IN (
    SELECT MIN(ROWID)
    FROM table_name
    GROUP BY unique_column
);
```

### 2. **FETCH FIRST (12c+)**
```sql
-- Oracle 12c and above
SELECT * FROM table_name
ORDER BY column_name
FETCH FIRST 10 ROWS ONLY;

-- Older versions
SELECT * FROM (
    SELECT * FROM table_name ORDER BY column_name
) WHERE ROWNUM <= 10;
```

### 3. **Connect By for Number Series**
```sql
-- Generate sequence without recursive CTE
SELECT LEVEL - 1 as n
FROM DUAL
CONNECT BY LEVEL <= 100;
```

---

## 🐍 Python Pandas Best Practices

### 1. **Memory Efficient Operations**
```python
# Use inplace operations for large datasets
df.drop_duplicates(subset='email', keep='first', inplace=True)

# Use vectorized operations
df['bonus'] = np.where(
    (df['id'] % 2 == 1) & (~df['name'].str.startswith('M')),
    df['salary'],
    0
)
```

### 2. **Date Operations**
```python
# Convert to datetime once
df['date'] = pd.to_datetime(df['date'])

# Extract components efficiently
df['year_month'] = df['date'].dt.to_period('M')
```

### 3. **Group Operations**
```python
# Use agg for multiple calculations
result = df.groupby('category').agg({
    'value': ['sum', 'mean', 'count'],
    'id': 'nunique'
})
```

---

## 📊 Performance Optimization Tips

### 1. **Index Usage**
```sql
-- Create indexes on commonly joined/filtered columns
CREATE INDEX idx_employee_dept ON Employee(departmentId);
CREATE INDEX idx_salary ON Employee(salary DESC);
```

### 2. **Avoid Functions on Indexed Columns**
```sql
-- ❌ Bad (prevents index usage)
WHERE UPPER(name) = 'JOHN'

-- ✅ Good
WHERE name = 'JOHN' OR name = 'john'
```

### 3. **Use EXISTS Instead of IN**
```sql
-- ❌ Slower for large datasets
WHERE id IN (SELECT id FROM other_table)

-- ✅ Faster
WHERE EXISTS (SELECT 1 FROM other_table WHERE id = main.id)
```

### 4. **Limit Data Early**
```sql
-- ❌ Bad
SELECT * FROM (
    SELECT * FROM large_table
    ORDER BY date DESC
) WHERE ROWNUM <= 100

-- ✅ Good
SELECT * FROM large_table
WHERE date >= SYSDATE - 30
ORDER BY date DESC
FETCH FIRST 100 ROWS ONLY
```

---

## 🧪 Testing Approach

### 1. **Edge Cases to Test**
- Empty tables
- Single row
- All duplicates
- NULL values
- Boundary dates
- Ties in rankings

### 2. **Sample Test Data**
```sql
-- Create test data
INSERT INTO Employee VALUES (1, 100, 1);
INSERT INTO Employee VALUES (2, 200, 1);
INSERT INTO Employee VALUES (3, 200, 2);  -- Tie case
INSERT INTO Employee VALUES (4, NULL, 1); -- NULL salary

-- Verify results
SELECT * FROM your_solution_view;
```

---

## 🎓 Study Tips

### Order of Learning
1. **Basic SELECT** → Easy problems (1757, 584, 595)
2. **JOINs** → Problems requiring multiple tables
3. **Aggregations** → GROUP BY problems
4. **Window Functions** → Ranking and analytics
5. **Advanced Patterns** → Consecutive, recursive

### Practice Strategy
1. Solve with SQL first
2. Implement in PL/SQL with error handling
3. Convert to Python Pandas
4. Compare performance
5. Optimize based on data size

---

## 📚 Quick Reference Commands

### Oracle SQL
```sql
-- Window functions
ROW_NUMBER() OVER (PARTITION BY col ORDER BY col2)
RANK() OVER (ORDER BY col)
DENSE_RANK() OVER (ORDER BY col)
LAG(col, n) OVER (ORDER BY col2)
LEAD(col, n) OVER (ORDER BY col2)

-- Date functions
SYSDATE
TRUNC(date_col)
ADD_MONTHS(date_col, n)
MONTHS_BETWEEN(date1, date2)

-- String functions
SUBSTR(str, start, length)
INSTR(str, substring)
REGEXP_LIKE(col, pattern)
LISTAGG(col, ',') WITHIN GROUP (ORDER BY col)
```

### Python Pandas
```python
# Window operations
df.groupby('category')['value'].rank(method='dense')
df['prev'] = df.groupby('id')['value'].shift(1)

# Date operations
pd.to_datetime(df['date'])
df['date'].dt.year
df['date'].dt.to_period('M')

# Aggregations
df.groupby('key').agg({'col': ['sum', 'mean']})
df.pivot_table(index='a', columns='b', values='c')
```

---

## 🔗 Related Resources

- **Oracle Documentation**: Window functions, analytic functions
- **Pandas Documentation**: GroupBy operations